# Clasificación de Textos Históricos — Notebook Unificado (Todos los Enfoques)
**Aprendizaje de Máquina 2026-10 · Universidad de los Andes**

## Instrucciones de uso

1. **Seleccionar modelo**: cambia `MODELO_A_EJECUTAR` en la celda de configuración.
2. **Correr en otra computadora**: solo necesitas este notebook y la carpeta `./data/`.
3. **Primera ejecución**: corre las secciones 0–5 (SETUP OBLIGATORIO) una sola vez.
   Luego puedes ejecutar cualquier sección de modelo de forma independiente.
4. **Predicciones**: se guardan en `./submissions/` si `val_score > 0.30` (F1-macro).
5. **Log de scores**: se actualiza en `./submissions/scores_log.csv` automáticamente.

### Modelos disponibles (valor de MODELO_A_EJECUTAR):
```
Clásicos:   'tfidf_logreg', 'ngram_lm', 'char_cnn_simple', 'svm', 'stylometry'
No-superv:  'topic_model', 'temporal_cluster', 'diachronic_w2v', 'semantic_drift', 'latent_temporal'
Transformers:'finetune_bert', 'dapt', 'tapt', 'ordinal_transformer', 'contrastive_sbert', 'multitask'
OCR:        'ocr_simulation', 'noise_injection', 'dual_pipeline', 'ocr_aware'
Ensemble:   'soft_voting', 'stacking', 'mixture_experts', 'calibration', 'temporal_smoothing'
Diagnóstico:'failure_modes'
Todos:      'ALL'  ← entrena todos en secuencia (muy lento)
```

## SECCIÓN 0 — Instalación de dependencias

In [1]:
import subprocess, sys
from pathlib import Path

def pip_install(*pkgs):
    for pkg in pkgs:
        print(f"  → instalando {pkg}...")
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(f"    FALLO: {r.stderr[-200:]}")

print("="*65)
print("INSTALADOR DE DEPENDENCIAS — Notebook Unificado")
print("="*65)

pip_install('numpy', 'pandas', 'scikit-learn', 'scipy', 'matplotlib', 'seaborn',
            'tqdm', 'joblib', 'lightgbm', 'gensim')

# PyTorch + Transformers (comentar si no tienes internet/GPU)
pip_install('torch', 'transformers', 'sentence-transformers', 'accelerate', 'sentencepiece')

print("\n✅ Instalación completa.")

INSTALADOR DE DEPENDENCIAS — Notebook Unificado
  → instalando numpy...
  → instalando pandas...
  → instalando scikit-learn...
  → instalando scipy...
  → instalando matplotlib...
  → instalando seaborn...
  → instalando tqdm...
  → instalando joblib...
  → instalando lightgbm...
  → instalando gensim...
  → instalando torch...
  → instalando transformers...
  → instalando sentence-transformers...
  → instalando accelerate...
  → instalando sentencepiece...

✅ Instalación completa.


## SECCIÓN 1 — Imports y configuración global

In [14]:
import os, re, json, random, warnings, unicodedata, pickle, time
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.linear_model import LogisticRegression as _SkLogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import CalibratedClassifierCV
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack as sparse_hstack, csr_matrix
import joblib

def LogisticRegression(*args, **kwargs):
    kwargs.pop('multi_class', None)
    return _SkLogisticRegression(*args, **kwargs)

try:
    import lightgbm as lgb
    LGBM_OK = True
except ImportError:
    LGBM_OK = False
    print("LightGBM no disponible.")

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader, TensorDataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                               AutoModel, get_linear_schedule_with_warmup,
                               DataCollatorWithPadding)
    TORCH_OK = True
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"PyTorch: {torch.__version__} | Device: {DEVICE}")
except ImportError:
    TORCH_OK = False
    DEVICE = None
    print("PyTorch/Transformers no disponibles. Modelos de transformer desactivados.")

try:
    from sentence_transformers import SentenceTransformer
    SBERT_OK = True
except ImportError:
    SBERT_OK = False

try:
    from gensim.models import Word2Vec, LdaModel
    from gensim.corpora import Dictionary
    GENSIM_OK = True
except ImportError:
    GENSIM_OK = False

warnings.filterwarnings('ignore')
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")

PyTorch: 2.12.0+cpu | Device: cpu
NumPy: 2.4.6 | Pandas: 3.0.3


In [3]:
# =============================================================================
# ██████  VARIABLE PRINCIPAL — CAMBIAR AQUÍ PARA SELECCIONAR MODELO ████████
# =============================================================================
MODELO_A_EJECUTAR = 'tfidf_logreg'
# Opciones: ver lista en la celda de instrucciones arriba.
# Para correr todos: MODELO_A_EJECUTAR = 'ALL'
# =============================================================================

SEED               = 42
VAL_SCORE_UMBRAL   = 0.30   # F1-macro mínimo para guardar predicciones
MODEL_THRESHOLD    = 0.30   # alias legible
ENSEMBLE_THRESHOLD = 0.25   # umbral más bajo para participar en ensemble

# Rutas — usa rutas relativas para portabilidad entre máquinas
DATA_DIR    = Path('./data')
PROCESS_DIR = Path('./process')
SUBMIT_DIR  = Path('./submissions')
MODEL_DIR   = Path('./models')

for d in [PROCESS_DIR/'pkl', PROCESS_DIR/'images', SUBMIT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ppath = lambda n: str(PROCESS_DIR / 'pkl' / n)
ipath = lambda n: str(PROCESS_DIR / 'images' / n)
spath = lambda n: str(SUBMIT_DIR / n)
mpath = lambda n: str(MODEL_DIR / n)

# Semillas de reproducibilidad
random.seed(SEED)
np.random.seed(SEED)
if TORCH_OK:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print(f"MODELO_A_EJECUTAR = '{MODELO_A_EJECUTAR}'")
print(f"Umbral val_score (F1-macro) = {VAL_SCORE_UMBRAL}")
print(f"Datos en: {DATA_DIR.resolve()}")

MODELO_A_EJECUTAR = 'tfidf_logreg'
Umbral val_score (F1-macro) = 0.3
Datos en: C:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\data


## SECCIÓN 2 — Funciones auxiliares: limpieza, features, evaluación

In [4]:
# ─── Mapa de caracteres históricos ─────────────────────────────────────────
_CHAR_MAP = str.maketrans({
    '\u017f': 's', '\u017F': 'S',   # long-s ſ
    '\ua75b': 'r', '\ua75B': 'R',   # rotunda-r ꝛ
    '\ufb01': 'fi', '\ufb02': 'fl', '\ufb00': 'ff',
    '\ufb03': 'ffi', '\ufb04': 'ffl', '\ufb05': 'st',
    '\u00e6': 'ae', '\u00c6': 'AE',
    '\u0153': 'oe', '\u0152': 'OE',
    '\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"',
    '\u25a0': ' ', '\u2022': ' ', '\u00ac': '',
    '\u00a0': ' ', '\u200b': '', '\u200c': '', '\u200d': '',
    '\ufffd': ' ', '\u00ad': '',
})

_OCR_FIXES = [
    (re.compile(r'\bfu\b'), 'su'), (re.compile(r'\bfe\b'), 'se'),
    (re.compile(r'\bfi\b'), 'si'), (re.compile(r'fla'), 'sla'),
    (re.compile(r'rn'), 'm'),
]

_META_RE   = re.compile(r'(?i)(copyright|google|digitized|archive\.org|hathitrust)[^\n]*\n?')
_URL_RE    = re.compile(r'https?://\S+|www\.\S+')
_WS_RE     = re.compile(r' {2,}')
_CTRL_RE   = re.compile(r'[\x00-\x08\x0e-\x1f\x7f]')
_LONG_S_RE = re.compile(r'\bf[aeiou]', re.I)


def clean_text_deep(text: str) -> str:
    """Limpieza profunda (text_norm): para TF-IDF y features lingüísticas."""
    if not isinstance(text, str):
        text = str(text)
    text = unicodedata.normalize('NFC', text)
    text = text.translate(_CHAR_MAP)
    text = re.sub(r'(?<=\w)[\-\u2010\u2011]\s*\n\s*(?=\w)', '', text)
    text = _META_RE.sub(' ', text)
    text = _URL_RE.sub(' ', text)
    text = _CTRL_RE.sub(' ', text)
    for pat, repl in _OCR_FIXES:
        text = pat.sub(repl, text)
    clean_lines = []
    for line in text.split('\n'):
        line = line.strip()
        if not line: continue
        letters = sum(c.isalpha() for c in line)
        if letters / max(len(line), 1) < 0.3 and len(line) > 20:
            continue
        clean_lines.append(line)
    text = ' '.join(clean_lines)
    text = _WS_RE.sub(' ', text).strip()
    return text


def clean_text_minimal(text: str) -> str:
    """Limpieza mínima (text_raw): preserva señal histórica ſ→f para transformers."""
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r'\s+', ' ', text).strip()


# ─── Features lingüísticas históricas ──────────────────────────────────────
RE_HAZER    = re.compile(r'\bha[zs]er\b|\bdi[zs]e\b|\bdixo\b|\btraxo\b', re.I)
RE_VOS      = re.compile(r'\bvos\b|\bvuestra\s+merced\b', re.I)
RE_CEDILLA  = re.compile(r'[çÇ]', re.I)
RE_NON      = re.compile(r'\bnon\b', re.I)
RE_AGORA    = re.compile(r'\bagora\b', re.I)
RE_AHORA    = re.compile(r'\bahora\b', re.I)
RE_MESMO    = re.compile(r'\bme[sz]mo\b', re.I)
RE_MISMO    = re.compile(r'\bmismo\b', re.I)
RE_MENTE    = re.compile(r'\w+mente\b', re.I)
RE_SUBORD   = re.compile(r'\b(que|cual|donde|cuando|porque|pues)\b', re.I)
RE_GERUND   = re.compile(r'\b\w+(ando|iendo)\b', re.I)
RE_LONG_S   = re.compile(r'\bf[aeiou]', re.I)

ARCAISMOS = {'ansí','assí','assi','muncho','agora','mesmo','mesma','ovo','desque','ca','fasta','primeramente'}
LEXICO_XVIII = {'razón','ilustración','filosofía','naturaleza','utilidad','progreso','nación','patria','reforma','educación'}
LEXICO_XIX = {'burgués','burguesía','ferrocarril','telégrafo','vapor','periódico','republicano','liberal','revolución','democracia'}
FUNC_WORDS = {'el','la','los','las','un','una','de','en','que','y','a','por','con','se','no','del','al','lo','le','su'}

def extract_handcrafted(texts):
    """Extrae ~30 features lingüísticas históricas. Retorna DataFrame."""
    rows = []
    for text in texts:
        if not isinstance(text, str) or len(text) == 0:
            rows.append({k: 0.0 for k in ['hazer','vos','cedilla','non','agora','ahora',
                'mesmo','mismo','mente','subord','gerund','long_s','arcaismos','lexico_xviii',
                'lexico_xix','n_words','avg_word_len','ttr','upper_ratio','punct_ratio',
                'comma_ratio','semicol_ratio','func_word_ratio','digit_ratio',
                'acc_vowels','sent_len_mean','sent_count','brackets','ellipsis','exclaim']})
            continue
        t, tl = text, text.lower()
        words = tl.split()
        nw = max(len(words), 1)
        nc = max(len(t), 1)
        sentences = re.split(r'[.!?]+', t)
        sent_lens = [len(s.split()) for s in sentences if s.strip()]
        rows.append({
            'hazer':        len(RE_HAZER.findall(t)) / nw,
            'vos':          len(RE_VOS.findall(t)) / nw,
            'cedilla':      len(RE_CEDILLA.findall(t)) / nc,
            'non':          len(RE_NON.findall(t)) / nw,
            'agora':        len(RE_AGORA.findall(t)) / nw,
            'ahora':        len(RE_AHORA.findall(t)) / nw,
            'mesmo':        len(RE_MESMO.findall(t)) / nw,
            'mismo':        len(RE_MISMO.findall(t)) / nw,
            'mente':        len(RE_MENTE.findall(t)) / nw,
            'subord':       len(RE_SUBORD.findall(t)) / nw,
            'gerund':       len(RE_GERUND.findall(t)) / nw,
            'long_s':       len(RE_LONG_S.findall(t)) / nw,
            'arcaismos':    len(set(words) & ARCAISMOS) / nw,
            'lexico_xviii': len(set(words) & LEXICO_XVIII) / nw,
            'lexico_xix':   len(set(words) & LEXICO_XIX) / nw,
            'n_words':      nw,
            'avg_word_len': nc / nw,
            'ttr':          len(set(words)) / nw,
            'upper_ratio':  sum(1 for c in t if c.isupper()) / nc,
            'punct_ratio':  sum(1 for c in t if c in '.,;:!?') / nc,
            'comma_ratio':  t.count(',') / nc,
            'semicol_ratio':t.count(';') / nc,
            'func_word_ratio': len([w for w in words if w in FUNC_WORDS]) / nw,
            'digit_ratio':  sum(c.isdigit() for c in t) / nc,
            'acc_vowels':   sum(1 for c in t if c in 'áéíóúàèìòùâêîôû') / nc,
            'sent_len_mean':np.mean(sent_lens) if sent_lens else 0,
            'sent_count':   len(sent_lens),
            'brackets':     t.count('(') / nc,
            'ellipsis':     t.count('...') / nc,
            'exclaim':      t.count('!') / nc,
        })
    return pd.DataFrame(rows).fillna(0).astype(np.float32)


# ─── Función de evaluación y guardado ──────────────────────────────────────
_scores_log = []  # registro global de scores

def evaluar_y_guardar(nombre_modelo, y_pred_val, y_pred_eval_decoded,
                      val_score=None, y_true_val=None, extra_info=""):
    """
    Evalúa val_score, muestra reporte, guarda submission si val_score > umbral.
    
    Parámetros:
        nombre_modelo        : str, nombre del modelo (para el archivo CSV)
        y_pred_val           : predicciones en validación (enteros codificados)
        y_pred_eval_decoded  : predicciones en eval final (décadas reales)
        val_score            : si ya calculado, úsalo; si None, calcula con y_true_val
        y_true_val           : labels reales de validación (solo si val_score=None)
        extra_info           : string adicional para el log
    Retorna:
        val_score (float)
    """
    global _scores_log, label_encoder, df_eval

    if val_score is None and y_true_val is not None:
        val_score = f1_score(y_true_val, y_pred_val, average='macro')
    elif val_score is None:
        raise ValueError("Proporciona val_score o y_true_val")

    print(f"\n{'='*60}")
    print(f"  Modelo: {nombre_modelo}")
    print(f"  F1-macro val: {val_score:.4f}  (umbral: {VAL_SCORE_UMBRAL})")
    print(f"{'='*60}")

    if y_true_val is not None:
        print(classification_report(
            y_true_val, y_pred_val,
            target_names=[str(c) for c in label_encoder.classes_],
            zero_division=0, digits=3
        )[:1500])  # recortamos para no inundar la salida

    if val_score > VAL_SCORE_UMBRAL:
        print(f"✅ val_score > {VAL_SCORE_UMBRAL} → generando submission...")
        fname = spath(f"predicciones_{nombre_modelo}.csv")
        sub = pd.DataFrame({'id': df_eval['id'], 'answer': y_pred_eval_decoded})
        sub.to_csv(fname, index=False)
        print(f"   Guardado: {fname}")
    else:
        print(f"❌ val_score <= {VAL_SCORE_UMBRAL} → NO se generan predicciones.")

    # Registrar en log
    entry = {'modelo': nombre_modelo, 'f1_macro_val': round(val_score, 4),
             'supera_umbral': val_score > VAL_SCORE_UMBRAL,
             'timestamp': pd.Timestamp.now().isoformat(), 'extra': extra_info}
    _scores_log.append(entry)

    log_path = spath('scores_log.csv')
    pd.DataFrame(_scores_log).to_csv(log_path, index=False)
    print(f"   Log actualizado: {log_path}")
    return val_score


# ─── Utilidad: ¿debe correr este modelo? ───────────────────────────────────
def debe_correr(nombre):
    return MODELO_A_EJECUTAR == 'ALL' or MODELO_A_EJECUTAR == nombre

## SECCIÓN 3 — SETUP OBLIGATORIO: Carga, limpieza, split y features base
> **Ejecuta esta sección UNA SOLA VEZ.** Todos los modelos dependen de las variables
> generadas aquí: `X_train`, `X_val`, `y_train`, `y_val`, etc.

In [5]:
# ─── 3.1 Carga de datos ────────────────────────────────────────────────────
print("Cargando datos...")
df_train = pd.read_csv(DATA_DIR / 'train.csv')
df_eval  = pd.read_csv(DATA_DIR / 'eval.csv')

print(f"Train : {df_train.shape}")
print(f"Eval  : {df_eval.shape}")
print(f"Cols train: {df_train.columns.tolist()}")
print(f"Décadas únicas: {sorted(df_train['decade'].unique())[:10]} ... (total: {df_train['decade'].nunique()})")

Cargando datos...
Train : (31403, 2)
Eval  : (3490, 2)
Cols train: ['text', 'decade']
Décadas únicas: [np.int64(150), np.int64(151), np.int64(152), np.int64(153), np.int64(154), np.int64(155), np.int64(156), np.int64(157), np.int64(158), np.int64(159)] ... (total: 39)


In [6]:
# ─── 3.2 EDA básico ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

decade_counts = df_train['decade'].value_counts().sort_index()
axes[0].bar(range(len(decade_counts)), decade_counts.values, color='steelblue', alpha=0.8)
axes[0].set_title('Distribución de décadas'); axes[0].set_xlabel('Índice década')

lens = df_train['text'].astype(str).str.len()
axes[1].hist(lens.clip(0, 5000), bins=60, color='steelblue', alpha=0.8)
axes[1].axvline(lens.mean(), color='red', linestyle='--', label=f'Media: {lens.mean():.0f}')
axes[1].set_title('Longitud de textos (chars)'); axes[1].legend()

mean_len = df_train.groupby('decade')['text'].apply(lambda x: x.astype(str).str.len().mean())
axes[2].plot(mean_len.index, mean_len.values, color='steelblue', marker='o', ms=3)
axes[2].set_title('Longitud media por década')

plt.tight_layout()
plt.savefig(ipath('eda_unificado.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f"Décadas: {df_train['decade'].nunique()} | Long. media: {lens.mean():.0f} | Imbalance: {decade_counts.max()/decade_counts.min():.1f}x")

Décadas: 39 | Long. media: 530 | Imbalance: 1.1x


In [7]:
# ─── 3.3 Limpieza dual (text_norm + text_raw) ──────────────────────────────
MIN_TEXT_LEN = 20

print("Aplicando limpieza dual...")
df_train['text_norm'] = df_train['text'].apply(clean_text_deep)
df_train['text_raw']  = df_train['text'].apply(clean_text_minimal)
df_eval['text_norm']  = df_eval['text'].apply(clean_text_deep)
df_eval['text_raw']   = df_eval['text'].apply(clean_text_minimal)

# Filtrar textos muy cortos en train
df_train = df_train.dropna(subset=['decade'])
df_train['decade'] = pd.to_numeric(df_train['decade'], errors='coerce')
df_train = df_train.dropna(subset=['decade']).reset_index(drop=True)
df_train['decade'] = df_train['decade'].astype(int)

mask = df_train['text_norm'].str.len() >= MIN_TEXT_LEN
n_removed = (~mask).sum()
df_train = df_train[mask].reset_index(drop=True)
print(f"Train después del filtro: {len(df_train)} ({n_removed} removidos por texto corto)")

Aplicando limpieza dual...
Train después del filtro: 31394 (9 removidos por texto corto)


In [15]:
# ─── 3.4 Codificación de labels + split temporal ──────────────────────────
label_encoder = LabelEncoder()
label_encoder.fit(df_train['decade'])
NUM_CLASSES = len(label_encoder.classes_)
DECADES = label_encoder.classes_

print(f"Clases (décadas): {NUM_CLASSES}")
print(f"Rango: {DECADES[0]} – {DECADES[-1]}")

with open(ppath('label_encoder.pkl'), 'wb') as f:
    pickle.dump(label_encoder, f)

# ── Split temporal estricto ────────────────────────────────────────────────
# IMPORTANTE: no usamos shuffle global para evitar temporal overfitting.
# Ordenamos por decade y hacemos un split proporcional estratificado.
# El 80% más "antiguo" de cada clase va a train, 10% val, 10% test interno.
# En la práctica, con ~30k ejemplos y ~39 clases, train_test_split estratificado
# es equivalente si el dataset no tiene estructura temporal interna obvia.
# Aquí ofrecemos ambas opciones:

TEMPORAL_SPLIT = False   # True = split temporal estricto por orden temporal; False = estratificado

if TEMPORAL_SPLIT:
    df_train_sorted = df_train.sort_values('decade').reset_index(drop=True)
    n = len(df_train_sorted)
    n_train = int(0.80 * n)
    n_val   = int(0.10 * n)
    df_tr  = df_train_sorted.iloc[:n_train]
    df_va  = df_train_sorted.iloc[n_train:n_train+n_val]
    df_te  = df_train_sorted.iloc[n_train+n_val:]
    X_train     = df_tr['text_norm'].values
    X_train_raw = df_tr['text_raw'].values
    y_train     = label_encoder.transform(df_tr['decade'])
    X_val       = df_va['text_norm'].values
    X_val_raw   = df_va['text_raw'].values
    y_val       = label_encoder.transform(df_va['decade'])
    X_test_int  = df_te['text_norm'].values
    X_test_raw  = df_te['text_raw'].values
    y_test_int  = label_encoder.transform(df_te['decade'])
    print("Split: TEMPORAL ESTRICTO (orden cronológico)")
else:
    y_all   = label_encoder.transform(df_train['decade'])
    X_all   = df_train['text_norm'].values
    X_all_r = df_train['text_raw'].values
    idx = np.arange(len(X_all))
    idx_tr, idx_tmp = train_test_split(idx, test_size=0.20, random_state=SEED, stratify=y_all)
    idx_va, idx_te  = train_test_split(idx_tmp, test_size=0.50, random_state=SEED, stratify=y_all[idx_tmp])

    X_train     = X_all[idx_tr];   X_train_raw = X_all_r[idx_tr];  y_train = y_all[idx_tr]
    X_val       = X_all[idx_va];   X_val_raw   = X_all_r[idx_va];  y_val   = y_all[idx_va]
    X_test_int  = X_all[idx_te];   X_test_raw  = X_all_r[idx_te];  y_test_int = y_all[idx_te]
    print("Split: ESTRATIFICADO (shuffle con semilla fija)")

X_eval      = df_eval['text_norm'].values
X_eval_raw  = df_eval['text_raw'].values

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test interno: {len(X_test_int)} | Eval: {len(X_eval)}")

# Pesos de clase para desbalance
cw_arr  = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
cw_dict = {i: w for i, w in enumerate(cw_arr)}

# One-hot para Keras (si se usa)
try:
    import tensorflow as tf
    y_train_oh    = tf.keras.utils.to_categorical(y_train,    NUM_CLASSES)
    y_val_oh      = tf.keras.utils.to_categorical(y_val,      NUM_CLASSES)
    y_test_int_oh = tf.keras.utils.to_categorical(y_test_int, NUM_CLASSES)
    TF_OK = True
except:
    TF_OK = False

Clases (décadas): 39
Rango: 150 – 188
Split: ESTRATIFICADO (shuffle con semilla fija)
Train: 25115 | Val: 3139 | Test interno: 3140 | Eval: 3490



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\.venv_ml\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\.venv_ml\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "c:\Users\User\Documents\

AttributeError: _ARRAY_API not found

ImportError: numpy.core._multiarray_umath failed to import

In [16]:
# ─── 3.5 TF-IDF base (word + char, necesarios para múltiples modelos) ─────
print("Construyendo TF-IDF word...")
tfidf_word = TfidfVectorizer(
    max_features=80_000, ngram_range=(1, 3), analyzer='word',
    sublinear_tf=True, min_df=2, max_df=0.97,
    token_pattern=r'(?u)\b[\w\u00C0-\u024F]{2,}\b',
)
Xtw_train = tfidf_word.fit_transform(X_train)
Xtw_val   = tfidf_word.transform(X_val)
Xtw_test  = tfidf_word.transform(X_test_int)
Xtw_eval  = tfidf_word.transform(X_eval)
joblib.dump(tfidf_word, ppath('tfidf_word.pkl'))
print(f"  Word TF-IDF: {Xtw_train.shape}")

print("Construyendo TF-IDF char...")
tfidf_char = TfidfVectorizer(
    max_features=50_000, ngram_range=(2, 6), analyzer='char_wb',
    sublinear_tf=True, min_df=3, max_df=0.95,
)
Xtc_train = tfidf_char.fit_transform(X_train)
Xtc_val   = tfidf_char.transform(X_val)
Xtc_test  = tfidf_char.transform(X_test_int)
Xtc_eval  = tfidf_char.transform(X_eval)
joblib.dump(tfidf_char, ppath('tfidf_char.pkl'))
print(f"  Char TF-IDF: {Xtc_train.shape}")

# Combinado
Xtcomb_train = sparse_hstack([Xtw_train, Xtc_train]).tocsr()
Xtcomb_val   = sparse_hstack([Xtw_val,   Xtc_val  ]).tocsr()
Xtcomb_test  = sparse_hstack([Xtw_test,  Xtc_test ]).tocsr()
Xtcomb_eval  = sparse_hstack([Xtw_eval,  Xtc_eval ]).tocsr()
print(f"  Combinado: {Xtcomb_train.shape}")

Construyendo TF-IDF word...
  Word TF-IDF: (25115, 80000)
Construyendo TF-IDF char...
  Char TF-IDF: (25115, 50000)
  Combinado: (25115, 130000)


In [17]:
# ─── 3.6 Handcrafted features (todas las secciones) ───────────────────────
print("Extrayendo features lingüísticas históricas...")
Xhc_train = extract_handcrafted(X_train.tolist())
Xhc_val   = extract_handcrafted(X_val.tolist())
Xhc_test  = extract_handcrafted(X_test_int.tolist())
Xhc_eval  = extract_handcrafted(X_eval.tolist())

scaler_hc = StandardScaler()
Xhc_train_sc = scaler_hc.fit_transform(Xhc_train)
Xhc_val_sc   = scaler_hc.transform(Xhc_val)
Xhc_test_sc  = scaler_hc.transform(Xhc_test)
Xhc_eval_sc  = scaler_hc.transform(Xhc_eval)
joblib.dump(scaler_hc, ppath('scaler_hc.pkl'))
print(f"  Handcrafted features: {Xhc_train_sc.shape}")

# ──── Registro de probabilidades de los modelos (para ensembles) ──────────
MODEL_PROBS = {}   # dict: nombre_modelo → (val_probs, eval_probs, val_score)

print("\n✅ SETUP COMPLETO. Ya puedes ejecutar cualquier sección de modelo.")
print(f"Variables disponibles: X_train ({len(X_train)}), X_val ({len(X_val)}), "
      f"y_train, y_val, Xtcomb_train, Xhc_train_sc, ...")

Extrayendo features lingüísticas históricas...
  Handcrafted features: (25115, 30)

✅ SETUP COMPLETO. Ya puedes ejecutar cualquier sección de modelo.
Variables disponibles: X_train (25115), X_val (3139), y_train, y_val, Xtcomb_train, Xhc_train_sc, ...


---
## SECCIÓN A — Métodos Clásicos

### A1. TF-IDF + Regresión Logística / SVM

In [18]:
if debe_correr('tfidf_logreg'):
    print("\n" + "="*60)
    print("MODELO A1: TF-IDF + Logistic Regression")
    print("="*60)

    # Grid sobre class_weight y C
    best_f1, best_clf, best_cw = -1, None, None
    for cw in [None, 'balanced']:
        for C in [0.1, 0.5, 1.0]:
            clf = LogisticRegression(
                C=C, max_iter=1000, solver='saga',
                class_weight=cw,
                random_state=SEED, n_jobs=-1
            )
            clf.fit(Xtcomb_train, y_train)
            f1 = f1_score(y_val, clf.predict(Xtcomb_val), average='macro')
            print(f"  class_weight={cw}, C={C} → F1-macro val = {f1:.4f}")
            if f1 > best_f1:
                best_f1, best_clf, best_cw = f1, clf, cw

    print(f"\nMejor configuración: class_weight={best_cw}, F1-macro={best_f1:.4f}")

    # Predicciones finales
    y_pred_val  = best_clf.predict(Xtcomb_val)
    y_pred_eval = label_encoder.inverse_transform(best_clf.predict(Xtcomb_eval))

    # Guardar probabilidades para ensemble
    if hasattr(best_clf, 'predict_proba'):
        MODEL_PROBS['tfidf_logreg'] = (
            best_clf.predict_proba(Xtcomb_val),
            best_clf.predict_proba(Xtcomb_eval),
            best_f1
        )

    joblib.dump(best_clf, mpath('tfidf_logreg.pkl'))
    evaluar_y_guardar('tfidf_logreg', y_pred_val, y_pred_eval,
                      val_score=best_f1, y_true_val=y_val)


MODELO A1: TF-IDF + Logistic Regression
  class_weight=None, C=0.1 → F1-macro val = 0.1966
  class_weight=None, C=0.5 → F1-macro val = 0.2493
  class_weight=None, C=1.0 → F1-macro val = 0.2583
  class_weight=balanced, C=0.1 → F1-macro val = 0.1966
  class_weight=balanced, C=0.5 → F1-macro val = 0.2476
  class_weight=balanced, C=1.0 → F1-macro val = 0.2584

Mejor configuración: class_weight=balanced, F1-macro=0.2584

  Modelo: tfidf_logreg
  F1-macro val: 0.2584  (umbral: 0.3)
              precision    recall  f1-score   support

         150      0.738     0.747     0.742        79
         151      0.456     0.704     0.553        81
         152      0.524     0.544     0.534        79
         153      0.490     0.628     0.551        78
         154      0.518     0.530     0.524        83
         155      0.413     0.313     0.356        83
         156      0.272     0.316     0.292        79
         157      0.260     0.301     0.279        83
         158      0.222     0.2

In [19]:
if debe_correr('svm'):
    print("\n" + "="*60)
    print("MODELO A4: TF-IDF + LinearSVC")
    print("="*60)

    # Usa Platt scaling para tener probabilidades
    base_svm = LinearSVC(C=0.5, max_iter=2000, class_weight='balanced', random_state=SEED)
    clf_svm  = CalibratedClassifierCV(base_svm, cv=3, method='sigmoid')
    clf_svm.fit(Xtcomb_train, y_train)

    y_pred_val  = clf_svm.predict(Xtcomb_val)
    val_f1      = f1_score(y_val, y_pred_val, average='macro')
    y_pred_eval = label_encoder.inverse_transform(clf_svm.predict(Xtcomb_eval))

    MODEL_PROBS['svm'] = (
        clf_svm.predict_proba(Xtcomb_val),
        clf_svm.predict_proba(Xtcomb_eval),
        val_f1
    )
    joblib.dump(clf_svm, mpath('svm.pkl'))
    evaluar_y_guardar('svm', y_pred_val, y_pred_eval,
                      val_score=val_f1, y_true_val=y_val)

In [20]:
if debe_correr('ngram_lm'):
    print("\n" + "="*60)
    print("MODELO A2: N-gram LM Features + Logistic Regression")
    print("="*60)

    # N-gramas de palabras (unigram, bigram, trigram) + N-gramas de caracteres
    # Se modela como clasificador sobre representaciones TF-IDF de n-gramas
    ngram_vecs = []
    for ngr in [(1,1),(2,2),(3,3)]:
        v = TfidfVectorizer(max_features=30_000, ngram_range=ngr, sublinear_tf=True,
                            min_df=2, analyzer='word')
        ngram_vecs.append((v, v.fit_transform(X_train), v.transform(X_val), v.transform(X_eval)))

    # Char n-gramas
    for ngr in [(2,4),(4,6)]:
        v = TfidfVectorizer(max_features=20_000, ngram_range=ngr, sublinear_tf=True,
                            min_df=3, analyzer='char_wb')
        ngram_vecs.append((v, v.fit_transform(X_train), v.transform(X_val), v.transform(X_eval)))

    X_ng_train = sparse_hstack([t for _,t,_,_ in ngram_vecs]).tocsr()
    X_ng_val   = sparse_hstack([v for _,_,v,_ in ngram_vecs]).tocsr()
    X_ng_eval  = sparse_hstack([e for _,_,_,e in ngram_vecs]).tocsr()

    clf_ng = LogisticRegression(C=0.3, max_iter=1000, solver='saga',
                                multi_class='multinomial', class_weight='balanced',
                                random_state=SEED, n_jobs=-1)
    clf_ng.fit(X_ng_train, y_train)
    y_pred_val  = clf_ng.predict(X_ng_val)
    val_f1      = f1_score(y_val, y_pred_val, average='macro')
    y_pred_eval = label_encoder.inverse_transform(clf_ng.predict(X_ng_eval))

    MODEL_PROBS['ngram_lm'] = (clf_ng.predict_proba(X_ng_val),
                               clf_ng.predict_proba(X_ng_eval), val_f1)
    evaluar_y_guardar('ngram_lm', y_pred_val, y_pred_eval,
                      val_score=val_f1, y_true_val=y_val)

In [21]:
if debe_correr('char_cnn_simple'):
    print("\n" + "="*60)
    print("MODELO A3: Character-level CNN (con Keras/PyTorch)")
    print("="*60)

    # ── Versión A: Hashing Vectorizer + LogReg (fallback rápido) ──────────
    hv = HashingVectorizer(analyzer='char_wb', ngram_range=(3,5), n_features=2**18,
                           norm='l2', alternate_sign=False)
    Xhash_tr  = hv.transform(X_train)
    Xhash_val = hv.transform(X_val)
    Xhash_ev  = hv.transform(X_eval)
    clf_ch = LinearSVC(C=0.3, max_iter=2000, class_weight='balanced', random_state=SEED)
    clf_ch.fit(Xhash_tr, y_train)
    y_pred_val = clf_ch.predict(Xhash_val)
    val_f1     = f1_score(y_val, y_pred_val, average='macro')
    y_pred_eval= label_encoder.inverse_transform(clf_ch.predict(Xhash_ev))
    evaluar_y_guardar('char_cnn_simple', y_pred_val, y_pred_eval,
                      val_score=val_f1, y_true_val=y_val,
                      extra_info='HashingVectorizer+LinearSVC (versión ligera)')

    # ── Versión B: CNN Keras (requiere TF) ────────────────────────────────
    if TF_OK:
        import tensorflow as tf
        CHAR_VOCAB = 200
        CHAR_LEN   = 1000

        def encode_chars(texts, max_len=CHAR_LEN):
            out = np.zeros((len(texts), max_len), dtype=np.int32)
            for i, t in enumerate(texts):
                for j, c in enumerate(t[:max_len]):
                    out[i, j] = min(ord(c), CHAR_VOCAB - 1)
            return out

        print("  Codificando caracteres (puede tardar ~1 min)...")
        Xch_tr  = encode_chars(X_train)
        Xch_val = encode_chars(X_val)
        Xch_ev  = encode_chars(X_eval)

        with tf.distribute.OneDeviceStrategy('/CPU:0').scope():
            inp = tf.keras.Input(shape=(CHAR_LEN,))
            x   = tf.keras.layers.Embedding(CHAR_VOCAB, 32)(inp)
            x   = tf.keras.layers.Conv1D(128, 5, activation='relu', padding='same')(x)
            x   = tf.keras.layers.GlobalMaxPooling1D()(x)
            x   = tf.keras.layers.Dense(128, activation='relu')(x)
            x   = tf.keras.layers.Dropout(0.3)(x)
            out = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
            model_cnn = tf.keras.Model(inp, out)

        model_cnn.compile(optimizer='adam',
                          loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                          metrics=['accuracy'])
        es = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True,
                                               monitor='val_loss')
        model_cnn.fit(Xch_tr, y_train, validation_data=(Xch_val, y_val),
                      epochs=30, batch_size=64, callbacks=[es], verbose=1,
                      class_weight=cw_dict)

        p_val  = model_cnn.predict(Xch_val,  verbose=0)
        p_eval = model_cnn.predict(Xch_ev,   verbose=0)
        val_f1_cnn = f1_score(y_val, p_val.argmax(1), average='macro')
        dec_eval   = label_encoder.inverse_transform(p_eval.argmax(1))

        MODEL_PROBS['char_cnn'] = (p_val, p_eval, val_f1_cnn)
        evaluar_y_guardar('char_cnn_keras', p_val.argmax(1), dec_eval,
                          val_score=val_f1_cnn, y_true_val=y_val,
                          extra_info='CharCNN Keras')

In [22]:
if debe_correr('stylometry'):
    print("\n" + "="*60)
    print("MODELO A5: Estilometría (features artesanales + LightGBM)")
    print("="*60)

    # Combinar handcrafted features con TF-IDF reducido (LSA 100d)
    lsa = TruncatedSVD(n_components=100, random_state=SEED)
    Xlsa_tr  = lsa.fit_transform(Xtcomb_train)
    Xlsa_val = lsa.transform(Xtcomb_val)
    Xlsa_ev  = lsa.transform(Xtcomb_eval)

    Xsty_tr  = np.hstack([Xhc_train_sc, Xlsa_tr])
    Xsty_val = np.hstack([Xhc_val_sc,   Xlsa_val])
    Xsty_ev  = np.hstack([Xhc_eval_sc,  Xlsa_ev])

    if LGBM_OK:
        clf_sty = lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=63,
            class_weight='balanced', random_state=SEED, n_jobs=-1,
            min_child_samples=5, subsample=0.8, colsample_bytree=0.8,
        )
        clf_sty.fit(Xsty_tr, y_train,
                    eval_set=[(Xsty_val, y_val)],
                    callbacks=[lgb.early_stopping(30, verbose=False)])
    else:
        clf_sty = LogisticRegression(C=1.0, max_iter=500, solver='saga',
                                     multi_class='multinomial', random_state=SEED, n_jobs=-1)
        clf_sty.fit(Xsty_tr, y_train)

    y_pred_val  = clf_sty.predict(Xsty_val)
    val_f1      = f1_score(y_val, y_pred_val, average='macro')
    y_pred_eval = label_encoder.inverse_transform(clf_sty.predict(Xsty_ev))

    proba_fn = clf_sty.predict_proba if hasattr(clf_sty, 'predict_proba') else None
    if proba_fn:
        MODEL_PROBS['stylometry'] = (proba_fn(Xsty_val), proba_fn(Xsty_ev), val_f1)
    evaluar_y_guardar('stylometry', y_pred_val, y_pred_eval,
                      val_score=val_f1, y_true_val=y_val)

---
## SECCIÓN B — Métodos No Supervisados

In [23]:
if debe_correr('topic_model'):
    print("\n" + "="*60)
    print("MODELO B1: Dynamic Topic Model (LDA) + Clasificador")
    print("="*60)
    # Extraemos representación de tópicos por documento y la usamos como feature

    N_TOPICS = 40  # ajustar según número de décadas

    if GENSIM_OK:
        print("  Usando Gensim LDA...")
        texts_tok = [t.lower().split() for t in X_train]
        dictionary = Dictionary(texts_tok)
        dictionary.filter_extremes(no_below=5, no_above=0.5)
        corpus_tr = [dictionary.doc2bow(t) for t in texts_tok]

        lda_model = LdaModel(corpus=corpus_tr, id2word=dictionary, num_topics=N_TOPICS,
                             passes=5, random_state=SEED, alpha='auto')

        def lda_features(texts, dict_, model, n_topics):
            feats = np.zeros((len(texts), n_topics), dtype=np.float32)
            for i, t in enumerate(texts):
                bow = dict_.doc2bow(t.lower().split())
                for tid, prob in model.get_document_topics(bow, minimum_probability=0.0):
                    feats[i, tid] = prob
            return feats

        Xtop_tr  = lda_features(X_train, dictionary, lda_model, N_TOPICS)
        Xtop_val = lda_features(X_val,   dictionary, lda_model, N_TOPICS)
        Xtop_ev  = lda_features(X_eval,  dictionary, lda_model, N_TOPICS)
    else:
        print("  Usando sklearn LDA...")
        lda_sk = LatentDirichletAllocation(n_components=N_TOPICS, max_iter=15,
                                            learning_method='online', random_state=SEED, n_jobs=-1)
        Xtop_tr  = lda_sk.fit_transform(Xtw_train).astype(np.float32)
        Xtop_val = lda_sk.transform(Xtw_val).astype(np.float32)
        Xtop_ev  = lda_sk.transform(Xtw_eval).astype(np.float32)

    # Combinar tópicos + handcrafted
    Xtm_tr  = np.hstack([Xtop_tr,  Xhc_train_sc])
    Xtm_val = np.hstack([Xtop_val, Xhc_val_sc  ])
    Xtm_ev  = np.hstack([Xtop_ev,  Xhc_eval_sc ])

    clf_tm = LogisticRegression(C=1.0, max_iter=500, solver='saga',
                                multi_class='multinomial', class_weight='balanced',
                                random_state=SEED, n_jobs=-1)
    clf_tm.fit(Xtm_tr, y_train)
    y_pred_val  = clf_tm.predict(Xtm_val)
    val_f1      = f1_score(y_val, y_pred_val, average='macro')
    y_pred_eval = label_encoder.inverse_transform(clf_tm.predict(Xtm_ev))

    MODEL_PROBS['topic_model'] = (clf_tm.predict_proba(Xtm_val),
                                   clf_tm.predict_proba(Xtm_ev), val_f1)
    evaluar_y_guardar('topic_model', y_pred_val, y_pred_eval,
                      val_score=val_f1, y_true_val=y_val)

In [24]:
if debe_correr('temporal_cluster'):
    print("\n" + "="*60)
    print("MODELO B2: Temporal Clustering por Décadas (features de embeddings)")
    print("="*60)
    # Reducimos con LSA, luego clusterizamos y usamos distancia al centroide como feature

    lsa_cl = TruncatedSVD(n_components=50, random_state=SEED)
    Xlsa_cl_tr  = lsa_cl.fit_transform(Xtcomb_train)
    Xlsa_cl_val = lsa_cl.transform(Xtcomb_val)
    Xlsa_cl_ev  = lsa_cl.transform(Xtcomb_eval)

    # Centroides por década (usando el conjunto de entrenamiento)
    centroids = {}
    for cls_idx in range(NUM_CLASSES):
        mask = y_train == cls_idx
        if mask.sum() > 0:
            centroids[cls_idx] = Xlsa_cl_tr[mask].mean(axis=0)

    def centroid_features(X_lsa, centroids, n_classes):
        dists = np.zeros((len(X_lsa), n_classes), dtype=np.float32)
        for cls_idx, c in centroids.items():
            dists[:, cls_idx] = np.linalg.norm(X_lsa - c, axis=1)
        return dists

    Xcd_tr  = centroid_features(Xlsa_cl_tr,  centroids, NUM_CLASSES)
    Xcd_val = centroid_features(Xlsa_cl_val, centroids, NUM_CLASSES)
    Xcd_ev  = centroid_features(Xlsa_cl_ev,  centroids, NUM_CLASSES)

    # El clasificador más simple: argmin distancia = clase predicha
    y_pred_nn  = Xcd_val.argmin(axis=1)
    y_pred_ev  = Xcd_ev.argmin(axis=1)
    val_f1_nn  = f1_score(y_val, y_pred_nn, average='macro')
    print(f"  Nearest-centroid F1-macro val: {val_f1_nn:.4f}")

    # Mejor con LogReg sobre distancias
    clf_cd = LogisticRegression(C=0.5, max_iter=500, solver='saga',
                                multi_class='multinomial', random_state=SEED, n_jobs=-1)
    clf_cd.fit(Xcd_tr, y_train)
    y_pred_val  = clf_cd.predict(Xcd_val)
    val_f1      = f1_score(y_val, y_pred_val, average='macro')
    y_pred_eval = label_encoder.inverse_transform(clf_cd.predict(Xcd_ev))

    MODEL_PROBS['temporal_cluster'] = (clf_cd.predict_proba(Xcd_val),
                                        clf_cd.predict_proba(Xcd_ev), val_f1)
    evaluar_y_guardar('temporal_cluster', y_pred_val, y_pred_eval,
                      val_score=val_f1, y_true_val=y_val)

In [25]:
if debe_correr('diachronic_w2v'):
    print("\n" + "="*60)
    print("MODELO B3: Diachronic Word2Vec (embeddings por siglo + alineación)")
    print("="*60)

    if not GENSIM_OK:
        print("  Requiere gensim. Saltando.")
    else:
        # Agrupar décadas en siglos (XV=1, XVI=2, ...) — ajustar al rango real
        df_full = df_train.copy()
        df_full['y_enc'] = y_train if not TEMPORAL_SPLIT else label_encoder.transform(df_full['decade'])
        df_full_all = df_train.copy()
        df_full_all['text_norm'] = X_all if not TEMPORAL_SPLIT else df_train['text_norm']
        df_full_all['y_enc'] = label_encoder.transform(df_train['decade'])

        century_map = {}
        for d in DECADES:
            century_map[d] = d // 100
        centuries = sorted(set(century_map.values()))
        print(f"  Siglos: {centuries}")

        # Entrenar Word2Vec por siglo
        w2v_models = {}
        for cent in centuries:
            mask = np.array([century_map[DECADES[y]] == cent for y in y_train])
            texts_c = [t.lower().split()[:200] for t in X_train[mask]]
            if len(texts_c) < 50:
                continue
            w2v = Word2Vec(sentences=texts_c, vector_size=100, window=5,
                           min_count=3, workers=4, epochs=5, seed=SEED)
            w2v_models[cent] = w2v
            print(f"    Siglo {cent}: {len(texts_c)} docs, vocab={len(w2v.wv)}")

        def doc_embedding_w2v(text, model, size=100):
            words = text.lower().split()
            vecs = [model.wv[w] for w in words if w in model.wv]
            return np.mean(vecs, axis=0) if vecs else np.zeros(size)

        # Feature: concatenar embeddings promedio de cada siglo
        EMB_DIM = 100
        def multi_century_emb(texts):
            features = np.zeros((len(texts), len(w2v_models) * EMB_DIM), dtype=np.float32)
            for j, (cent, model) in enumerate(sorted(w2v_models.items())):
                for i, t in enumerate(texts):
                    features[i, j*EMB_DIM:(j+1)*EMB_DIM] = doc_embedding_w2v(t, model, EMB_DIM)
            return features

        print("  Calculando embeddings multi-siglo...")
        Xw2v_tr  = multi_century_emb(X_train)
        Xw2v_val = multi_century_emb(X_val)
        Xw2v_ev  = multi_century_emb(X_eval)

        clf_w2v = LogisticRegression(C=1.0, max_iter=500, solver='saga',
                                     multi_class='multinomial', class_weight='balanced',
                                     random_state=SEED, n_jobs=-1)
        clf_w2v.fit(Xw2v_tr, y_train)
        y_pred_val  = clf_w2v.predict(Xw2v_val)
        val_f1      = f1_score(y_val, y_pred_val, average='macro')
        y_pred_eval = label_encoder.inverse_transform(clf_w2v.predict(Xw2v_ev))

        MODEL_PROBS['diachronic_w2v'] = (clf_w2v.predict_proba(Xw2v_val),
                                          clf_w2v.predict_proba(Xw2v_ev), val_f1)
        evaluar_y_guardar('diachronic_w2v', y_pred_val, y_pred_eval,
                          val_score=val_f1, y_true_val=y_val)

In [26]:
if debe_correr('semantic_drift'):
    print("\n" + "="*60)
    print("MODELO B4: Temporal Semantic Drift como feature")
    print("="*60)
    # Calcula distancia entre word embedding de cada texto y centroides temporales de LSA
    # como proxy del drift semántico

    # Usamos la representación LSA ya calculada (si no existe, la recalculamos)
    try:
        Xlsa_d
    except NameError:
        lsa_d = TruncatedSVD(n_components=30, random_state=SEED)
        Xlsa_d_tr  = lsa_d.fit_transform(Xtcomb_train)
        Xlsa_d_val = lsa_d.transform(Xtcomb_val)
        Xlsa_d_ev  = lsa_d.transform(Xtcomb_eval)
    else:
        Xlsa_d_tr, Xlsa_d_val, Xlsa_d_ev = Xlsa_d, Xlsa_d, Xlsa_d

    # Feature de drift: diferencia entre embedding del documento y el centroide de su
    # vecino temporal más cercano vs. el más lejano
    lsa_d = TruncatedSVD(n_components=30, random_state=SEED)
    Xlsa_d_tr  = lsa_d.fit_transform(Xtcomb_train)
    Xlsa_d_val = lsa_d.transform(Xtcomb_val)
    Xlsa_d_ev  = lsa_d.transform(Xtcomb_eval)

    # Centroides LSA por clase (décadas ordenadas)
    cent_lsa = np.zeros((NUM_CLASSES, 30), dtype=np.float32)
    for c in range(NUM_CLASSES):
        mask = y_train == c
        if mask.sum() > 0:
            cent_lsa[c] = Xlsa_d_tr[mask].mean(axis=0)

    def drift_features(X_lsa, cent):
        # Para cada doc: distancia al centroide más cercano, más lejano, promedio,
        # índice del más cercano (normalizado)
        dists = np.sqrt(((X_lsa[:, None, :] - cent[None, :, :]) ** 2).sum(axis=2))  # (N, C)
        return np.column_stack([
            dists.min(axis=1),          # distancia al más cercano
            dists.max(axis=1),          # distancia al más lejano
            dists.mean(axis=1),         # distancia media
            dists.argmin(axis=1) / NUM_CLASSES,  # índice relativo del más cercano
        ])

    Xdrift_tr  = drift_features(Xlsa_d_tr,  cent_lsa)
    Xdrift_val = drift_features(Xlsa_d_val, cent_lsa)
    Xdrift_ev  = drift_features(Xlsa_d_ev,  cent_lsa)

    Xsd_tr  = np.hstack([Xdrift_tr,  Xhc_train_sc, Xlsa_d_tr])
    Xsd_val = np.hstack([Xdrift_val, Xhc_val_sc,   Xlsa_d_val])
    Xsd_ev  = np.hstack([Xdrift_ev,  Xhc_eval_sc,  Xlsa_d_ev])

    clf_sd = LogisticRegression(C=1.0, max_iter=500, solver='saga',
                                multi_class='multinomial', class_weight='balanced',
                                random_state=SEED, n_jobs=-1)
    clf_sd.fit(Xsd_tr, y_train)
    y_pred_val  = clf_sd.predict(Xsd_val)
    val_f1      = f1_score(y_val, y_pred_val, average='macro')
    y_pred_eval = label_encoder.inverse_transform(clf_sd.predict(Xsd_ev))
    evaluar_y_guardar('semantic_drift', y_pred_val, y_pred_eval,
                      val_score=val_f1, y_true_val=y_val)

In [27]:
if debe_correr('latent_temporal'):
    print("\n" + "="*60)
    print("MODELO B5: Latent Temporal Space (Autoencoder con regularización temporal)")
    print("="*60)

    if not TORCH_OK:
        print("  Requiere PyTorch. Saltando.")
    else:
        # Autoencoder simple sobre representación LSA con loss de reconstrucción
        # + regularización que penaliza distancia entre docs cercanos en el tiempo

        lsa_ae = TruncatedSVD(n_components=100, random_state=SEED)
        Xae_tr  = lsa_ae.fit_transform(Xtcomb_train).astype(np.float32)
        Xae_val = lsa_ae.transform(Xtcomb_val).astype(np.float32)
        Xae_ev  = lsa_ae.transform(Xtcomb_eval).astype(np.float32)

        class TemporalAutoencoder(nn.Module):
            def __init__(self, in_dim=100, lat_dim=32):
                super().__init__()
                self.enc = nn.Sequential(
                    nn.Linear(in_dim, 64), nn.ReLU(),
                    nn.Linear(64, lat_dim),
                )
                self.dec = nn.Sequential(
                    nn.Linear(lat_dim, 64), nn.ReLU(),
                    nn.Linear(64, in_dim),
                )
            def forward(self, x):
                z = self.enc(x)
                return self.dec(z), z

        model_ae = TemporalAutoencoder(100, 32).to(DEVICE)
        optimizer_ae = torch.optim.Adam(model_ae.parameters(), lr=1e-3)

        Xae_T = torch.from_numpy(Xae_tr).to(DEVICE)
        y_T   = torch.from_numpy(y_train.astype(np.float32)).to(DEVICE)
        ds_ae = TensorDataset(Xae_T, y_T)
        dl_ae = DataLoader(ds_ae, batch_size=256, shuffle=True)

        print("  Entrenando autoencoder temporal (10 epochs)...")
        model_ae.train()
        for ep in range(10):
            total_loss = 0
            for xb, yb in dl_ae:
                xb = xb.to(DEVICE)
                recon, z = model_ae(xb)
                loss_recon = F.mse_loss(recon, xb)
                # Regularización temporal: pares con clase adyacente deben estar cerca
                loss_temp  = torch.tensor(0.0, device=DEVICE)
                if len(z) > 1:
                    y_diff = (yb[1:] - yb[:-1]).abs()
                    z_diff = (z[1:] - z[:-1]).norm(dim=1)
                    # Si son adyacentes en tiempo (diff <= 2), penalizar separación
                    adj = (y_diff <= 2).float()
                    loss_temp = (adj * z_diff).mean()
                loss = loss_recon + 0.1 * loss_temp
                optimizer_ae.zero_grad()
                loss.backward()
                optimizer_ae.step()
                total_loss += loss.item()
            if ep % 3 == 0:
                print(f"    Epoch {ep}: loss={total_loss/len(dl_ae):.4f}")

        model_ae.eval()
        def get_latent(X_np):
            with torch.no_grad():
                return model_ae.enc(torch.from_numpy(X_np).to(DEVICE)).cpu().numpy()

        Zlat_tr  = get_latent(Xae_tr)
        Zlat_val = get_latent(Xae_val)
        Zlat_ev  = get_latent(Xae_ev)

        Xlt_tr  = np.hstack([Zlat_tr,  Xhc_train_sc])
        Xlt_val = np.hstack([Zlat_val, Xhc_val_sc  ])
        Xlt_ev  = np.hstack([Zlat_ev,  Xhc_eval_sc ])

        clf_lt = LogisticRegression(C=1.0, max_iter=500, solver='saga',
                                    multi_class='multinomial', class_weight='balanced',
                                    random_state=SEED, n_jobs=-1)
        clf_lt.fit(Xlt_tr, y_train)
        y_pred_val  = clf_lt.predict(Xlt_val)
        val_f1      = f1_score(y_val, y_pred_val, average='macro')
        y_pred_eval = label_encoder.inverse_transform(clf_lt.predict(Xlt_ev))
        MODEL_PROBS['latent_temporal'] = (clf_lt.predict_proba(Xlt_val),
                                           clf_lt.predict_proba(Xlt_ev), val_f1)
        evaluar_y_guardar('latent_temporal', y_pred_val, y_pred_eval,
                          val_score=val_f1, y_true_val=y_val)

---
## SECCIÓN C — Transformers

In [28]:
# ─── C.1 Dataset PyTorch para transformers ─────────────────────────────────
if TORCH_OK:
    class TextDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_len=256):
            self.texts = texts
            self.labels = labels
            self.tok = tokenizer
            self.max_len = max_len

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            enc = self.tok(
                self.texts[idx], max_length=self.max_len,
                truncation=True, padding='max_length', return_tensors='pt'
            )
            return {k: v.squeeze(0) for k, v in enc.items()}, \
                   torch.tensor(self.labels[idx], dtype=torch.long)

    def train_transformer(model, train_ds, val_ds, n_epochs=5, batch_size=16,
                          lr=2e-5, warmup_ratio=0.1, grad_clip=1.0):
        """Bucle de entrenamiento con warmup + early stopping."""
        train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_dl   = DataLoader(val_ds,   batch_size=batch_size*2, shuffle=False)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = len(train_dl) * n_epochs
        warmup_steps = int(total_steps * warmup_ratio)
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

        best_f1, best_state = 0.0, None
        patience, patience_cnt = 2, 0

        for ep in range(n_epochs):
            model.train()
            total_loss = 0
            for batch, labels in tqdm(train_dl, desc=f'Epoch {ep+1}', leave=False):
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                labels = labels.to(DEVICE)
                out  = model(**batch, labels=labels)
                loss = out.loss
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                scheduler.step()
                total_loss += loss.item()

            # Validación
            model.eval()
            all_preds, all_labels = [], []
            with torch.no_grad():
                for batch, labels in val_dl:
                    batch = {k: v.to(DEVICE) for k, v in batch.items()}
                    out = model(**batch)
                    preds = out.logits.argmax(dim=-1).cpu().numpy()
                    all_preds.extend(preds)
                    all_labels.extend(labels.numpy())

            f1 = f1_score(all_labels, all_preds, average='macro')
            print(f"  Epoch {ep+1}: loss={total_loss/len(train_dl):.4f} | val F1-macro={f1:.4f}")

            if f1 > best_f1:
                best_f1 = f1
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_cnt = 0
            else:
                patience_cnt += 1
                if patience_cnt >= patience:
                    print(f"  Early stopping en epoch {ep+1}")
                    break

        model.load_state_dict(best_state)
        return model, best_f1

    def get_probs_transformer(model, texts, tokenizer, max_len=256, batch_size=32):
        """Extrae probabilidades del transformers sobre un conjunto de textos."""
        model.eval()
        all_probs = []
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            enc = tokenizer(batch_texts, max_length=max_len, truncation=True,
                            padding=True, return_tensors='pt')
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            with torch.no_grad():
                out = model(**enc)
            probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
        return np.vstack(all_probs)

In [29]:
if debe_correr('finetune_bert'):
    print("\n" + "="*60)
    print("MODELO C1: Fine-tuning XLM-RoBERTa-small")
    print("="*60)

    if not TORCH_OK:
        print("  Requiere PyTorch. Saltando.")
    else:
        MODEL_NAME = 'xlm-roberta-base'
        # Para recursos limitados, usar 'distilbert-base-multilingual-cased'
        # o 'PlanTL-GOB-ES/roberta-base-bne' para español
        MAX_LEN_BERT = 256

        print(f"  Cargando tokenizer: {MODEL_NAME}")
        tok_bert = AutoTokenizer.from_pretrained(MODEL_NAME)

        # Limitar tamaño de train para velocidad (quitar el [:5000] para entrenamiento completo)
        BERT_TRAIN_SIZE = min(len(X_train_raw), 8000)
        idx_bert = np.random.choice(len(X_train_raw), BERT_TRAIN_SIZE, replace=False)

        train_ds = TextDataset(X_train_raw[idx_bert].tolist(), y_train[idx_bert], tok_bert, MAX_LEN_BERT)
        val_ds   = TextDataset(X_val_raw.tolist(), y_val, tok_bert, MAX_LEN_BERT)

        model_bert = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
        ).to(DEVICE)

        model_bert, best_f1 = train_transformer(
            model_bert, train_ds, val_ds,
            n_epochs=5, batch_size=16 if torch.cuda.is_available() else 8,
            lr=2e-5, warmup_ratio=0.1
        )

        p_val  = get_probs_transformer(model_bert, X_val_raw.tolist(), tok_bert, MAX_LEN_BERT)
        p_eval = get_probs_transformer(model_bert, X_eval_raw.tolist(), tok_bert, MAX_LEN_BERT)

        y_pred_val  = p_val.argmax(1)
        val_f1      = f1_score(y_val, y_pred_val, average='macro')
        y_pred_eval = label_encoder.inverse_transform(p_eval.argmax(1))

        torch.save(model_bert.state_dict(), mpath('finetune_bert.pt'))
        MODEL_PROBS['finetune_bert'] = (p_val, p_eval, val_f1)
        evaluar_y_guardar('finetune_bert', y_pred_val, y_pred_eval,
                          val_score=val_f1, y_true_val=y_val)

In [30]:
if debe_correr('dapt'):
    print("\n" + "="*60)
    print("MODELO C2: DAPT — Domain-Adaptive Pretraining (versión reducida)")
    print("="*60)
    print("""
  Implementación completa de DAPT requiere varios GB y horas de entrenamiento.
  Aquí se proporciona una versión simulada con entrenamiento MLM en 1 época
  sobre una muestra del corpus histórico.
  
  Para escalar:
    - Usa la librería 'run_mlm.py' de HuggingFace con --num_train_epochs 3-5
    - Comando: python run_mlm.py --model_name_or_path xlm-roberta-base
               --train_file ./data/all_texts.txt --output_dir ./models/dapt
               --num_train_epochs 3 --per_device_train_batch_size 8
  """)

    if not TORCH_OK:
        print("  Requiere PyTorch. Saltando.")
    else:
        from transformers import (AutoModelForMaskedLM, DataCollatorForLanguageModeling,
                                   Trainer, TrainingArguments)

        MODEL_NAME = 'xlm-roberta-base'
        tok_dapt = AutoTokenizer.from_pretrained(MODEL_NAME)

        # Preparar dataset de textos (muestra pequeña para demo)
        sample_texts = np.random.choice(X_train_raw, min(500, len(X_train_raw)), replace=False).tolist()

        class MLMDataset(Dataset):
            def __init__(self, texts, tokenizer, max_len=256):
                self.enc = tokenizer(texts, max_length=max_len, truncation=True,
                                     padding='max_length', return_tensors='pt')
            def __len__(self): return self.enc['input_ids'].shape[0]
            def __getitem__(self, i):
                return {k: v[i] for k, v in self.enc.items()}

        mlm_ds = MLMDataset(sample_texts, tok_dapt)
        collator = DataCollatorForLanguageModeling(tok_dapt, mlm_probability=0.15)

        model_mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)
        train_args = TrainingArguments(
            output_dir=mpath('dapt_mlm'), overwrite_output_dir=True,
            num_train_epochs=1, per_device_train_batch_size=8,
            save_steps=1000, logging_steps=50, fp16=torch.cuda.is_available(),
            report_to='none',
        )
        trainer = Trainer(model=model_mlm, args=train_args,
                          train_dataset=mlm_ds, data_collator=collator)
        print("  Ejecutando 1 época de MLM pretraining (muestra 500 docs)...")
        trainer.train()
        model_mlm.save_pretrained(mpath('dapt_pretrained'))
        tok_dapt.save_pretrained(mpath('dapt_pretrained'))
        print("  Modelo DAPT guardado. Ahora fine-tuning de clasificación...")

        # Fine-tune el modelo DAPT para clasificación
        train_ds_d = TextDataset(X_train_raw.tolist()[:3000], y_train[:3000], tok_dapt, 256)
        val_ds_d   = TextDataset(X_val_raw.tolist(),            y_val,          tok_dapt, 256)
        model_dapt = AutoModelForSequenceClassification.from_pretrained(
            mpath('dapt_pretrained'), num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
        ).to(DEVICE)
        model_dapt, best_f1 = train_transformer(model_dapt, train_ds_d, val_ds_d, n_epochs=3)

        p_val  = get_probs_transformer(model_dapt, X_val_raw.tolist(), tok_dapt)
        p_eval = get_probs_transformer(model_dapt, X_eval_raw.tolist(), tok_dapt)
        y_pred_val  = p_val.argmax(1)
        val_f1      = f1_score(y_val, y_pred_val, average='macro')
        y_pred_eval = label_encoder.inverse_transform(p_eval.argmax(1))
        MODEL_PROBS['dapt'] = (p_val, p_eval, val_f1)
        evaluar_y_guardar('dapt', y_pred_val, y_pred_eval,
                          val_score=val_f1, y_true_val=y_val)

In [31]:
if debe_correr('tapt'):
    print("\n" + "="*60)
    print("MODELO C3: TAPT — Task-Adaptive Pretraining")
    print("="*60)
    print("""
  TAPT == DAPT pero usando exclusivamente los datos de la tarea (train.csv).
  Diferencia clave vs DAPT: el corpus de pretraining es el mismo que el de
  fine-tuning (self-supervised sobre textos históricos del dataset).
  Misma implementación que DAPT pero con todos los textos de entrenamiento.
  (Requiere las mismas herramientas que C2.)
  """)
    # Implementación idéntica a DAPT pero se usa todo el corpus de train
    print("  Para ejecutar TAPT, ejecuta la celda DAPT (C2) — es el mismo flujo.")
    print("  La diferencia está en el corpus de pretraining, no en el código.")

In [32]:
if debe_correr('ordinal_transformer'):
    print("\n" + "="*60)
    print("MODELO C4: Ordinal Transformer con CORAL Loss")
    print("="*60)

    if not TORCH_OK:
        print("  Requiere PyTorch. Saltando.")
    else:
        # CORAL (COnsistent RAnk Logits) - implementación propia
        # Referencia: Cao et al. 2020 - Rank Consistent Ordinal Regression

        class CoralLoss(nn.Module):
            """CORAL ordinal regression loss."""
            def __init__(self, num_classes):
                super().__init__()
                self.K = num_classes - 1

            def forward(self, logits, y):
                # logits: (N, K), y: (N,) enteros
                # Convertir a etiquetas binarias ordenadas
                rank_targets = torch.zeros(len(y), self.K, device=logits.device)
                for k in range(self.K):
                    rank_targets[:, k] = (y > k).float()
                return F.binary_cross_entropy_with_logits(logits, rank_targets)

        class OrdinalTransformer(nn.Module):
            def __init__(self, encoder, hidden_dim, num_classes):
                super().__init__()
                self.encoder = encoder
                # Cabeza ordinal: K-1 umbrales
                self.ord_head = nn.Linear(hidden_dim, num_classes - 1)
                self.bias = nn.Parameter(torch.zeros(num_classes - 1))

            def forward(self, **kwargs):
                out    = self.encoder(**kwargs)
                pooled = out.last_hidden_state[:, 0, :]  # [CLS]
                logits = self.ord_head(pooled) + self.bias
                return logits

            def predict(self, logits):
                probs_cumulative = torch.sigmoid(logits)
                # P(y=k) = P(y>k-1) - P(y>k)
                probs = torch.diff(
                    torch.cat([torch.zeros(len(logits),1,device=logits.device),
                               probs_cumulative,
                               torch.ones(len(logits),1,device=logits.device)], dim=1),
                    dim=1
                ).clamp(min=1e-7)
                return probs / probs.sum(dim=1, keepdim=True)

        MODEL_NAME = 'distilbert-base-multilingual-cased'
        tok_ord = AutoTokenizer.from_pretrained(MODEL_NAME)
        encoder_ord = AutoModel.from_pretrained(MODEL_NAME)
        hidden_dim  = encoder_ord.config.hidden_size

        model_ord = OrdinalTransformer(encoder_ord, hidden_dim, NUM_CLASSES).to(DEVICE)
        coral_loss = CoralLoss(NUM_CLASSES)
        optimizer_ord = torch.optim.AdamW(model_ord.parameters(), lr=2e-5)

        # Dataset compacto
        SUBSET = min(4000, len(X_train_raw))
        idx_ord = np.random.choice(len(X_train_raw), SUBSET, replace=False)

        def ord_collate(texts, labels, tokenizer, max_len=256):
            enc = tokenizer(texts, max_length=max_len, truncation=True,
                            padding=True, return_tensors='pt')
            return enc, torch.tensor(labels, dtype=torch.long)

        from torch.utils.data import random_split
        train_ds_ord = TextDataset(X_train_raw[idx_ord].tolist(), y_train[idx_ord], tok_ord, 256)
        val_ds_ord   = TextDataset(X_val_raw.tolist(), y_val, tok_ord, 256)
        dl_ord_tr    = DataLoader(train_ds_ord, batch_size=16, shuffle=True)
        dl_ord_val   = DataLoader(val_ds_ord,   batch_size=32, shuffle=False)

        best_f1_ord, best_state_ord = 0.0, None
        print("  Entrenando Ordinal Transformer (3 epochs)...")
        for ep in range(3):
            model_ord.train()
            for batch, labels in tqdm(dl_ord_tr, desc=f'Epoch {ep+1}', leave=False):
                batch  = {k: v.to(DEVICE) for k, v in batch.items()}
                labels = labels.to(DEVICE)
                logits = model_ord(**batch)
                loss   = coral_loss(logits, labels)
                optimizer_ord.zero_grad()
                loss.backward()
                optimizer_ord.step()

            model_ord.eval()
            all_preds, all_labs = [], []
            with torch.no_grad():
                for batch, labels in dl_ord_val:
                    batch = {k: v.to(DEVICE) for k, v in batch.items()}
                    logits = model_ord(**batch)
                    probs  = model_ord.predict(logits).cpu().numpy()
                    all_preds.extend(probs.argmax(1))
                    all_labs.extend(labels.numpy())
            f1 = f1_score(all_labs, all_preds, average='macro')
            print(f"    Epoch {ep+1}: F1-macro val = {f1:.4f}")
            if f1 > best_f1_ord:
                best_f1_ord = f1
                best_state_ord = {k: v.cpu().clone() for k, v in model_ord.state_dict().items()}

        model_ord.load_state_dict(best_state_ord)
        model_ord.eval()

        def probs_ordinal(model, texts, tokenizer, batch_size=32):
            all_p = []
            for i in range(0, len(texts), batch_size):
                enc = tokenizer(texts[i:i+batch_size], max_length=256,
                                truncation=True, padding=True, return_tensors='pt')
                enc = {k: v.to(DEVICE) for k, v in enc.items()}
                with torch.no_grad():
                    log = model(**enc)
                    p   = model.predict(log).cpu().numpy()
                all_p.append(p)
            return np.vstack(all_p)

        p_val  = probs_ordinal(model_ord, X_val_raw.tolist(),  tok_ord)
        p_eval = probs_ordinal(model_ord, X_eval_raw.tolist(), tok_ord)
        y_pred_val  = p_val.argmax(1)
        val_f1      = f1_score(y_val, y_pred_val, average='macro')
        y_pred_eval = label_encoder.inverse_transform(p_eval.argmax(1))
        MODEL_PROBS['ordinal_transformer'] = (p_val, p_eval, val_f1)
        evaluar_y_guardar('ordinal_transformer', y_pred_val, y_pred_eval,
                          val_score=val_f1, y_true_val=y_val)

In [33]:
if debe_correr('contrastive_sbert'):
    print("\n" + "="*60)
    print("MODELO C5: Contrastive Temporal Learning (Sentence-BERT)")
    print("="*60)

    if not (TORCH_OK and SBERT_OK):
        print("  Requiere sentence-transformers. Saltando.")
    else:
        # Extraer embeddings con SentenceBERT
        model_sbert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

        SBERT_SAMPLE = min(5000, len(X_train_raw))
        idx_sb = np.random.choice(len(X_train_raw), SBERT_SAMPLE, replace=False)

        print("  Generando embeddings SBERT (train)...")
        Xsb_tr = model_sbert.encode(X_train_raw[idx_sb].tolist(),
                                    batch_size=64, show_progress_bar=True,
                                    convert_to_numpy=True)
        print("  Generando embeddings SBERT (val)...")
        Xsb_val = model_sbert.encode(X_val_raw.tolist(),
                                     batch_size=64, show_progress_bar=True,
                                     convert_to_numpy=True)
        print("  Generando embeddings SBERT (eval)...")
        Xsb_ev = model_sbert.encode(X_eval_raw.tolist(),
                                    batch_size=64, show_progress_bar=True,
                                    convert_to_numpy=True)

        np.save(ppath('sbert_train.npy'), Xsb_tr)
        np.save(ppath('sbert_val.npy'),   Xsb_val)
        np.save(ppath('sbert_eval.npy'),  Xsb_ev)

        # Clasificador sobre embeddings SBERT
        clf_sb = LogisticRegression(C=1.0, max_iter=500, solver='lbfgs',
                                    multi_class='multinomial', class_weight='balanced',
                                    random_state=SEED, n_jobs=-1)
        clf_sb.fit(Xsb_tr, y_train[idx_sb])
        y_pred_val  = clf_sb.predict(Xsb_val)
        val_f1      = f1_score(y_val, y_pred_val, average='macro')
        y_pred_eval = label_encoder.inverse_transform(clf_sb.predict(Xsb_ev))

        MODEL_PROBS['contrastive_sbert'] = (clf_sb.predict_proba(Xsb_val),
                                             clf_sb.predict_proba(Xsb_ev), val_f1)
        evaluar_y_guardar('contrastive_sbert', y_pred_val, y_pred_eval,
                          val_score=val_f1, y_true_val=y_val,
                          extra_info='SBERT embeddings + LogReg')

In [34]:
if debe_correr('multitask'):
    print("\n" + "="*60)
    print("MODELO C6: Multi-task Temporal (predice década + siglo)")
    print("="*60)

    if not TORCH_OK:
        print("  Requiere PyTorch. Saltando.")
    else:
        # Tarea auxiliar: predecir siglo (agrupa décadas)
        century_labels_train = np.array([DECADES[y] // 100 for y in y_train])
        century_labels_val   = np.array([DECADES[y] // 100 for y in y_val])
        century_enc = LabelEncoder()
        century_enc.fit(np.concatenate([century_labels_train, century_labels_val]))
        NUM_CENTURY = len(century_enc.classes_)
        yc_train = century_enc.transform(century_labels_train)
        yc_val   = century_enc.transform(century_labels_val)

        class MultiTaskDataset(Dataset):
            def __init__(self, texts, y_decade, y_century, tokenizer, max_len=128):
                enc = tokenizer(texts, max_length=max_len, truncation=True,
                                padding='max_length', return_tensors='pt')
                self.enc   = enc
                self.yd    = torch.tensor(y_decade,  dtype=torch.long)
                self.yc    = torch.tensor(y_century, dtype=torch.long)
            def __len__(self): return len(self.yd)
            def __getitem__(self, i):
                return {k: v[i] for k, v in self.enc.items()}, self.yd[i], self.yc[i]

        MODEL_NAME = 'distilbert-base-multilingual-cased'
        tok_mt = AutoTokenizer.from_pretrained(MODEL_NAME)

        SUBSET_MT = min(4000, len(X_train_raw))
        idx_mt = np.random.choice(len(X_train_raw), SUBSET_MT, replace=False)

        train_ds_mt = MultiTaskDataset(X_train_raw[idx_mt].tolist(),
                                        y_train[idx_mt], yc_train[idx_mt], tok_mt)
        val_ds_mt   = MultiTaskDataset(X_val_raw.tolist(), y_val, yc_val, tok_mt)
        dl_mt_tr    = DataLoader(train_ds_mt, batch_size=16, shuffle=True)
        dl_mt_val   = DataLoader(val_ds_mt,   batch_size=32, shuffle=False)

        encoder_mt = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
        hidden_dim = encoder_mt.config.hidden_size
        head_dec   = nn.Linear(hidden_dim, NUM_CLASSES).to(DEVICE)
        head_cen   = nn.Linear(hidden_dim, NUM_CENTURY).to(DEVICE)
        params_mt  = list(encoder_mt.parameters()) + list(head_dec.parameters()) + list(head_cen.parameters())
        opt_mt     = torch.optim.AdamW(params_mt, lr=2e-5)

        best_f1_mt, best_states_mt = 0.0, None
        print("  Entrenando Multi-task (3 epochs)...")
        for ep in range(3):
            encoder_mt.train(); head_dec.train(); head_cen.train()
            for batch, yd, yc in tqdm(dl_mt_tr, desc=f'Epoch {ep+1}', leave=False):
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                yd, yc = yd.to(DEVICE), yc.to(DEVICE)
                h    = encoder_mt(**batch).last_hidden_state[:, 0, :]
                loss = F.cross_entropy(head_dec(h), yd) + 0.3 * F.cross_entropy(head_cen(h), yc)
                opt_mt.zero_grad()
                loss.backward()
                opt_mt.step()

            encoder_mt.eval(); head_dec.eval()
            preds, labs = [], []
            with torch.no_grad():
                for batch, yd, yc in dl_mt_val:
                    batch = {k: v.to(DEVICE) for k, v in batch.items()}
                    h = encoder_mt(**batch).last_hidden_state[:, 0, :]
                    p = head_dec(h).argmax(1).cpu().numpy()
                    preds.extend(p); labs.extend(yd.numpy())
            f1 = f1_score(labs, preds, average='macro')
            print(f"    Epoch {ep+1}: val F1={f1:.4f}")
            if f1 > best_f1_mt:
                best_f1_mt = f1
                best_states_mt = {
                    'enc': {k: v.cpu().clone() for k,v in encoder_mt.state_dict().items()},
                    'dec': {k: v.cpu().clone() for k,v in head_dec.state_dict().items()},
                }

        encoder_mt.load_state_dict(best_states_mt['enc'])
        head_dec.load_state_dict(best_states_mt['dec'])
        encoder_mt.eval(); head_dec.eval()

        def probs_mt(texts, tokenizer, enc, head, batch_size=32):
            all_p = []
            for i in range(0, len(texts), batch_size):
                enc_ = tokenizer(texts[i:i+batch_size], max_length=128,
                                 truncation=True, padding=True, return_tensors='pt')
                enc_ = {k: v.to(DEVICE) for k, v in enc_.items()}
                with torch.no_grad():
                    h = enc(**enc_).last_hidden_state[:, 0, :]
                    p = torch.softmax(head(h), dim=-1).cpu().numpy()
                all_p.append(p)
            return np.vstack(all_p)

        p_val  = probs_mt(X_val_raw.tolist(),  tok_mt, encoder_mt, head_dec)
        p_eval = probs_mt(X_eval_raw.tolist(), tok_mt, encoder_mt, head_dec)
        y_pred_val  = p_val.argmax(1)
        val_f1      = f1_score(y_val, y_pred_val, average='macro')
        y_pred_eval = label_encoder.inverse_transform(p_eval.argmax(1))
        MODEL_PROBS['multitask'] = (p_val, p_eval, val_f1)
        evaluar_y_guardar('multitask', y_pred_val, y_pred_eval,
                          val_score=val_f1, y_true_val=y_val)

---
## SECCIÓN D — Técnicas específicas para OCR histórico

In [35]:
if debe_correr('ocr_simulation') or debe_correr('noise_injection'):
    print("\n" + "="*60)
    print("MODELO D1+D2: OCR Simulation + Noise Injection (Data Augmentation)")
    print("="*60)

    # Confusiones OCR características de textos históricos
    OCR_CONFUSIONS = {
        's': ['f', 'ſ', '6'],   # long-s
        'u': ['n', 'v'],         # u/v indiferenciados
        'v': ['u', 'b'],
        'n': ['m', 'u'],
        'm': ['n', 'rn'],
        'o': ['0', 'a'],
        'i': ['l', '1'],
        'c': ['e', 'o'],
        'e': ['c', 'a'],
        't': ['l', 'f'],
        'h': ['li'],
    }

    def inject_ocr_noise(text, p_char=0.03, p_word=0.05):
        """Simula ruido OCR: confusiones de caracteres y dropout de palabras."""
        chars = list(text)
        for i, c in enumerate(chars):
            if random.random() < p_char and c.lower() in OCR_CONFUSIONS:
                chars[i] = random.choice(OCR_CONFUSIONS[c.lower()])
        text = ''.join(chars)
        words = text.split()
        words = [w for w in words if random.random() > p_word]
        return ' '.join(words)

    # Aumentar datos de train con ruido OCR
    X_train_noisy = np.array([inject_ocr_noise(t) for t in X_train])

    # Combinar original + ruidoso
    X_aug_all = np.concatenate([X_train, X_train_noisy])
    y_aug_all = np.concatenate([y_train, y_train])

    # Reentrenar TF-IDF con datos aumentados
    tfidf_noise = TfidfVectorizer(max_features=80_000, ngram_range=(1,3),
                                   sublinear_tf=True, min_df=2, analyzer='word')
    Xn_tr_aug = tfidf_noise.fit_transform(X_aug_all)
    Xn_val    = tfidf_noise.transform(X_val)
    Xn_ev     = tfidf_noise.transform(X_eval)

    clf_n = LogisticRegression(C=0.5, max_iter=1000, solver='saga',
                               multi_class='multinomial', class_weight='balanced',
                               random_state=SEED, n_jobs=-1)
    clf_n.fit(Xn_tr_aug, y_aug_all)
    y_pred_val  = clf_n.predict(Xn_val)
    val_f1      = f1_score(y_val, y_pred_val, average='macro')
    y_pred_eval = label_encoder.inverse_transform(clf_n.predict(Xn_ev))
    MODEL_PROBS['noise_injection'] = (clf_n.predict_proba(Xn_val),
                                       clf_n.predict_proba(Xn_ev), val_f1)
    evaluar_y_guardar('noise_injection', y_pred_val, y_pred_eval,
                      val_score=val_f1, y_true_val=y_val,
                      extra_info='TF-IDF entrenado con datos + ruido OCR')

In [36]:
if debe_correr('dual_pipeline'):
    print("\n" + "="*60)
    print("MODELO D3: Dual Pipeline (texto crudo + normalizado → fusión)")
    print("="*60)

    # Rama 1: TF-IDF sobre text_norm (normalizado)
    tf_norm = TfidfVectorizer(max_features=60_000, ngram_range=(1,3), sublinear_tf=True, min_df=2)
    Xn_tr = tf_norm.fit_transform(X_train)
    Xn_va = tf_norm.transform(X_val)
    Xn_ev = tf_norm.transform(X_eval)

    # Rama 2: TF-IDF sobre text_raw (preserva ſ→f señal histórica)
    tf_raw = TfidfVectorizer(max_features=60_000, ngram_range=(1,3), sublinear_tf=True, min_df=2)
    Xr_tr = tf_raw.fit_transform(X_train_raw)
    Xr_va = tf_raw.transform(X_val_raw)
    Xr_ev = tf_raw.transform(X_eval_raw)

    # Fusión: concatenar ambas ramas
    Xdp_tr = sparse_hstack([Xn_tr, Xr_tr]).tocsr()
    Xdp_va = sparse_hstack([Xn_va, Xr_va]).tocsr()
    Xdp_ev = sparse_hstack([Xn_ev, Xr_ev]).tocsr()

    clf_dp = LogisticRegression(C=0.5, max_iter=1000, solver='saga',
                                multi_class='multinomial', class_weight='balanced',
                                random_state=SEED, n_jobs=-1)
    clf_dp.fit(Xdp_tr, y_train)
    y_pred_val  = clf_dp.predict(Xdp_va)
    val_f1      = f1_score(y_val, y_pred_val, average='macro')
    y_pred_eval = label_encoder.inverse_transform(clf_dp.predict(Xdp_ev))

    MODEL_PROBS['dual_pipeline'] = (clf_dp.predict_proba(Xdp_va),
                                     clf_dp.predict_proba(Xdp_ev), val_f1)
    evaluar_y_guardar('dual_pipeline', y_pred_val, y_pred_eval,
                      val_score=val_f1, y_true_val=y_val)

In [37]:
if debe_correr('ocr_aware'):
    print("\n" + "="*60)
    print("MODELO D4: OCR-Aware Embeddings (features de calidad OCR)")
    print("="*60)

    # Métricas de calidad OCR como features adicionales
    def ocr_quality_features(texts):
        """Cuantifica el ruido OCR del texto."""
        rows = []
        for text in texts:
            if not isinstance(text, str) or len(text) < 5:
                rows.append([0.0] * 8); continue
            words = text.split()
            nw = max(len(words), 1)
            nc = max(len(text), 1)
            # Proporción de palabras que parecen OCR confundido (long-s)
            long_s = sum(1 for w in words if re.match(r'^f[aeiou]', w, re.I)) / nw
            # Palabras no-diccionario (heurística: muchas consonantes juntas)
            noisy_words = sum(1 for w in words if len(re.findall(r'[bcdfghjklmnpqrstvwxyz]{4,}', w.lower())) > 0) / nw
            # Densidad de símbolos especiales
            symbols = sum(1 for c in text if not c.isalnum() and not c.isspace()) / nc
            # Densidad de dígitos
            digits  = sum(c.isdigit() for c in text) / nc
            # Proporción de palabras de 1 char (artefactos)
            single_char = sum(1 for w in words if len(w) == 1) / nw
            # Palabras con mayúsculas intercaladas (artefacto OCR)
            mixed_case = sum(1 for w in words if len(w) > 2 and w != w.lower() and w != w.upper()) / nw
            # Varianza de longitud de palabras
            wl  = [len(w) for w in words]
            wl_std = np.std(wl) if len(wl) > 1 else 0
            # Repetición de secuencias de puntuación
            punct_seq = len(re.findall(r'[.,;:!?]{2,}', text)) / nc
            rows.append([long_s, noisy_words, symbols, digits, single_char,
                         mixed_case, wl_std, punct_seq])
        return np.array(rows, dtype=np.float32)

    Xocr_tr  = ocr_quality_features(X_train.tolist())
    Xocr_val = ocr_quality_features(X_val.tolist())
    Xocr_ev  = ocr_quality_features(X_eval.tolist())

    # Combinar features OCR + handcrafted + LSA-TF-IDF
    lsa_ocr = TruncatedSVD(n_components=50, random_state=SEED)
    Xlsa_ocr_tr  = lsa_ocr.fit_transform(Xtcomb_train)
    Xlsa_ocr_val = lsa_ocr.transform(Xtcomb_val)
    Xlsa_ocr_ev  = lsa_ocr.transform(Xtcomb_eval)

    Xoa_tr  = np.hstack([Xlsa_ocr_tr,  Xhc_train_sc, Xocr_tr])
    Xoa_val = np.hstack([Xlsa_ocr_val, Xhc_val_sc,   Xocr_val])
    Xoa_ev  = np.hstack([Xlsa_ocr_ev,  Xhc_eval_sc,  Xocr_ev])

    clf_oa = LogisticRegression(C=1.0, max_iter=500, solver='saga',
                                multi_class='multinomial', class_weight='balanced',
                                random_state=SEED, n_jobs=-1)
    clf_oa.fit(Xoa_tr, y_train)
    y_pred_val  = clf_oa.predict(Xoa_val)
    val_f1      = f1_score(y_val, y_pred_val, average='macro')
    y_pred_eval = label_encoder.inverse_transform(clf_oa.predict(Xoa_ev))
    MODEL_PROBS['ocr_aware'] = (clf_oa.predict_proba(Xoa_val),
                                 clf_oa.predict_proba(Xoa_ev), val_f1)
    evaluar_y_guardar('ocr_aware', y_pred_val, y_pred_eval,
                      val_score=val_f1, y_true_val=y_val)

---
## SECCIÓN E — Ensembles

In [38]:
if debe_correr('soft_voting') or debe_correr('ALL'):
    print("\n" + "="*60)
    print("MODELO E1: Soft Voting Ensemble")
    print("="*60)

    if len(MODEL_PROBS) < 2:
        print("  ⚠️  Necesitas al menos 2 modelos entrenados con probabilidades.")
        print("  Ejecuta primero algunos modelos de las secciones A-D.")
    else:
        # Filtrar modelos que superan el umbral del ensemble
        eligible = {n: (pv, pe, s) for n, (pv, pe, s) in MODEL_PROBS.items()
                    if s >= ENSEMBLE_THRESHOLD}
        print(f"  Modelos elegibles ({len(eligible)}): {list(eligible.keys())}")

        if len(eligible) < 2:
            print("  No hay suficientes modelos elegibles. Bajando umbral...")
            eligible = dict(list(MODEL_PROBS.items())[:min(5, len(MODEL_PROBS))])

        # Promedio simple de probabilidades
        p_val_ens  = np.mean([pv for pv, _, _ in eligible.values()], axis=0)
        p_eval_ens = np.mean([pe for _, pe, _ in eligible.values()], axis=0)

        y_pred_val  = p_val_ens.argmax(1)
        val_f1      = f1_score(y_val, y_pred_val, average='macro')
        y_pred_eval = label_encoder.inverse_transform(p_eval_ens.argmax(1))

        MODEL_PROBS['soft_voting'] = (p_val_ens, p_eval_ens, val_f1)
        evaluar_y_guardar('soft_voting', y_pred_val, y_pred_eval,
                          val_score=val_f1, y_true_val=y_val,
                          extra_info=f"Modelos: {list(eligible.keys())}")

In [39]:
if debe_correr('stacking') or debe_correr('ALL'):
    print("\n" + "="*60)
    print("MODELO E2: Stacking (meta-clasificador)")
    print("="*60)

    if len(MODEL_PROBS) < 2:
        print("  ⚠️  Necesitas al menos 2 modelos. Saltando.")
    else:
        eligible = {n: (pv, pe, s) for n, (pv, pe, s) in MODEL_PROBS.items()
                    if s >= ENSEMBLE_THRESHOLD and 'soft_voting' not in n and 'stacking' not in n}

        if len(eligible) < 2:
            eligible = {k: v for k, v in list(MODEL_PROBS.items())[:5]
                        if 'soft_voting' not in k}

        # Meta-features: concatenar probabilidades de modelos base
        X_meta_val  = np.hstack([pv for pv, _, _ in eligible.values()])
        X_meta_eval = np.hstack([pe for _, pe, _ in eligible.values()])

        # Para el meta-train, usamos cross-val en validación (out-of-fold)
        # Simplificado: usamos las probs de val como proxy (puede sobreestimar)
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        oof_proba = np.zeros((len(y_val), NUM_CLASSES), dtype=np.float32)
        meta_clf  = LogisticRegression(C=0.5, max_iter=200, solver='lbfgs',
                                        multi_class='multinomial', random_state=SEED)
        for tr_i, va_i in skf.split(X_meta_val, y_val):
            meta_clf.fit(X_meta_val[tr_i], y_val[tr_i])
            oof_proba[va_i] = meta_clf.predict_proba(X_meta_val[va_i])

        meta_clf.fit(X_meta_val, y_val)  # refit en todo val para predecir eval
        p_meta_eval  = meta_clf.predict_proba(X_meta_eval)

        y_pred_val  = oof_proba.argmax(1)
        val_f1      = f1_score(y_val, y_pred_val, average='macro')
        y_pred_eval = label_encoder.inverse_transform(p_meta_eval.argmax(1))
        MODEL_PROBS['stacking'] = (oof_proba, p_meta_eval, val_f1)
        evaluar_y_guardar('stacking', y_pred_val, y_pred_eval,
                          val_score=val_f1, y_true_val=y_val,
                          extra_info=f"Meta-clf sobre: {list(eligible.keys())}")

In [40]:
if debe_correr('mixture_experts') or debe_correr('ALL'):
    print("\n" + "="*60)
    print("MODELO E3: Mixture of Experts (Gating Network)")
    print("="*60)

    if len(MODEL_PROBS) < 2:
        print("  ⚠️  Necesitas al menos 2 modelos. Saltando.")
    else:
        eligible = {n: (pv, pe, s) for n, (pv, pe, s) in MODEL_PROBS.items()
                    if 'soft_voting' not in n and 'stacking' not in n and
                       'mixture' not in n}
        N_EXPERTS = min(len(eligible), 5)
        exp_names = list(eligible.keys())[:N_EXPERTS]

        # Gating network: LogReg que aprende qué experto confiar en cada ejemplo
        # Input: handcrafted features (señal sobre el tipo de texto)
        # Output: pesos sobre los expertos

        if not TORCH_OK:
            # Versión sklearn del gating network
            gating = LogisticRegression(C=1.0, max_iter=300, solver='lbfgs',
                                        multi_class='multinomial', random_state=SEED)
            gating.fit(Xhc_train_sc, y_train)
            gate_probs_val  = gating.predict_proba(Xhc_val_sc)   # (N_val, NUM_CLASSES)
            gate_probs_eval = gating.predict_proba(Xhc_eval_sc)

            # Los pesos de expertos se estiman como similitud entre gate y predicción de cada experto
            p_moe_val  = np.zeros_like(list(eligible.values())[0][0])
            p_moe_eval = np.zeros_like(list(eligible.values())[0][1])
            for i, name in enumerate(exp_names):
                pv, pe, s = eligible[name]
                # Peso = correlación entre gate y experto (simplificado: promedio ponderado por score)
                p_moe_val  += s * pv
                p_moe_eval += s * pe
            total_w = sum(eligible[n][2] for n in exp_names)
            p_moe_val  /= total_w
            p_moe_eval /= total_w
        else:
            # Gating con red pequeña sobre handcrafted features
            class GatingNet(nn.Module):
                def __init__(self, in_dim, n_experts):
                    super().__init__()
                    self.net = nn.Sequential(
                        nn.Linear(in_dim, 64), nn.ReLU(), nn.Dropout(0.2),
                        nn.Linear(64, n_experts), nn.Softmax(dim=-1)
                    )
                def forward(self, x): return self.net(x)

            gate_net = GatingNet(Xhc_train_sc.shape[1], N_EXPERTS).to(DEVICE)
            # Compilar listas de probs de expertos
            exp_probs_val  = np.stack([eligible[n][0] for n in exp_names], axis=0)  # (E, N, C)
            exp_probs_eval = np.stack([eligible[n][1] for n in exp_names], axis=0)

            Xg_tr = torch.from_numpy(Xhc_train_sc).float().to(DEVICE)
            yg_tr = torch.from_numpy(y_train).long().to(DEVICE)
            Xg_va = torch.from_numpy(Xhc_val_sc).float().to(DEVICE)

            # Entrenamos el gating minimizando la NLL de la mezcla sobre val
            # (más correcto: usar OOF, aquí simplificamos)
            opt_g = torch.optim.Adam(gate_net.parameters(), lr=1e-3)
            # Aproximación: minimizar NLL sobre train usando probs de expertos interpoladas
            # (train no tiene probs de expertos; usamos las val como proxy en mini-batches)
            ep_probs_va_T = torch.from_numpy(exp_probs_val.transpose(1,0,2)).float().to(DEVICE)  # (N_val, E, C)
            yg_va = torch.from_numpy(y_val).long().to(DEVICE)
            dl_g  = DataLoader(TensorDataset(ep_probs_va_T, yg_va), batch_size=64, shuffle=True)

            gate_net.train()
            for ep in range(20):
                for xb, yb in dl_g:
                    gate_w  = gate_net(Xg_va[:len(xb)]).unsqueeze(2)  # (B, E, 1)
                    mix     = (gate_w * xb).sum(dim=1)                 # (B, C)
                    loss    = F.nll_loss(torch.log(mix + 1e-8), yb[:len(xb)])
                    opt_g.zero_grad(); loss.backward(); opt_g.step()

            gate_net.eval()
            with torch.no_grad():
                gw_val  = gate_net(Xg_va).cpu().numpy()           # (N_val, E)
                gw_eval = gate_net(torch.from_numpy(Xhc_eval_sc).float().to(DEVICE)).cpu().numpy()

            p_moe_val  = np.einsum('ne,nec->nc', gw_val,  exp_probs_val.T.reshape(len(y_val),N_EXPERTS,NUM_CLASSES) if False else np.stack([eligible[n][0] for n in exp_names], axis=1).transpose(0,1,2))
            # Simplificado con einsum correcto:
            p_moe_val  = np.stack([eligible[n][0] for n in exp_names], axis=-1)  # (N,C,E)
            p_moe_val  = (p_moe_val * gw_val[:, np.newaxis, :]).sum(axis=-1)     # (N,C)

            p_moe_eval = np.stack([eligible[n][1] for n in exp_names], axis=-1)
            p_moe_eval = (p_moe_eval * gw_eval[:, np.newaxis, :]).sum(axis=-1)

        y_pred_val  = p_moe_val.argmax(1)
        val_f1      = f1_score(y_val, y_pred_val, average='macro')
        y_pred_eval = label_encoder.inverse_transform(p_moe_eval.argmax(1))
        MODEL_PROBS['mixture_experts'] = (p_moe_val, p_moe_eval, val_f1)
        evaluar_y_guardar('mixture_experts', y_pred_val, y_pred_eval,
                          val_score=val_f1, y_true_val=y_val,
                          extra_info=f"Expertos: {exp_names}")

In [41]:
if debe_correr('calibration') or debe_correr('ALL'):
    print("\n" + "="*60)
    print("MODELO E4: Calibración (Isotonic Regression + Platt Scaling)")
    print("="*60)

    if len(MODEL_PROBS) == 0:
        print("  ⚠️  Sin modelos entrenados. Saltando.")
    else:
        from sklearn.isotonic import IsotonicRegression

        # Tomar el mejor modelo disponible y calibrar sus probs
        best_model_name = max(MODEL_PROBS, key=lambda k: MODEL_PROBS[k][2])
        p_val_bc, p_eval_bc, score_bc = MODEL_PROBS[best_model_name]
        print(f"  Calibrando: {best_model_name} (F1-macro base: {score_bc:.4f})")

        # Calibración por clase (isotonic regression)
        # Método: para cada clase, fit IsotonicRegression sobre las probs de val
        p_cal_val  = p_val_bc.copy()
        p_cal_eval = p_eval_bc.copy()

        for c in range(NUM_CLASSES):
            y_bin = (y_val == c).astype(int)
            if y_bin.sum() < 5:
                continue
            iso = IsotonicRegression(out_of_bounds='clip')
            try:
                iso.fit(p_val_bc[:, c], y_bin)
                p_cal_val[:, c]  = iso.predict(p_val_bc[:, c])
                p_cal_eval[:, c] = iso.predict(p_eval_bc[:, c])
            except:
                pass

        # Renormalizar
        p_cal_val  /= p_cal_val.sum(axis=1, keepdims=True)
        p_cal_eval /= p_cal_eval.sum(axis=1, keepdims=True)

        y_pred_val  = p_cal_val.argmax(1)
        val_f1      = f1_score(y_val, y_pred_val, average='macro')
        y_pred_eval = label_encoder.inverse_transform(p_cal_eval.argmax(1))

        MODEL_PROBS['calibration'] = (p_cal_val, p_cal_eval, val_f1)
        evaluar_y_guardar('calibration', y_pred_val, y_pred_eval,
                          val_score=val_f1, y_true_val=y_val,
                          extra_info=f"Calibrado sobre: {best_model_name}")

In [42]:
if debe_correr('temporal_smoothing') or debe_correr('ALL'):
    print("\n" + "="*60)
    print("MODELO E5: Temporal Smoothing (promedio móvil de predicciones)")
    print("="*60)

    if len(MODEL_PROBS) == 0:
        print("  ⚠️  Sin modelos entrenados. Saltando.")
    else:
        best_model_name = max(MODEL_PROBS, key=lambda k: MODEL_PROBS[k][2]
                              if 'smooth' not in k else 0)
        p_val_ts, p_eval_ts, _ = MODEL_PROBS[best_model_name]

        # Suavizado temporal: convolucion 1D sobre las K clases (ordenadas temporalmente)
        # Asumimos que las clases están ordenadas por década (label_encoder las ordena)
        def temporal_smooth(probs, sigma=0.5):
            """Aplica kernel gaussiano sobre la distribución de probabilidades (clases ordenadas)."""
            K = probs.shape[1]
            x = np.arange(K)
            kernel = np.exp(-0.5 * (x / sigma) ** 2)
            kernel = kernel / kernel.sum()
            # Aplicar como convolución por filas
            smoothed = np.apply_along_axis(
                lambda row: np.convolve(row, kernel, mode='same'), axis=1, arr=probs
            )
            return smoothed / smoothed.sum(axis=1, keepdims=True)

        best_sigma = 0.0
        best_f1_sm = f1_score(y_val, p_val_ts.argmax(1), average='macro')

        for sigma in [0.3, 0.5, 1.0, 1.5, 2.0]:
            p_sm = temporal_smooth(p_val_ts, sigma)
            f1_sm = f1_score(y_val, p_sm.argmax(1), average='macro')
            print(f"  sigma={sigma}: F1-macro = {f1_sm:.4f}")
            if f1_sm > best_f1_sm:
                best_f1_sm = f1_sm
                best_sigma = sigma

        if best_sigma > 0:
            p_val_best  = temporal_smooth(p_val_ts,  best_sigma)
            p_eval_best = temporal_smooth(p_eval_ts, best_sigma)
        else:
            p_val_best, p_eval_best = p_val_ts, p_eval_ts

        print(f"  Mejor sigma = {best_sigma}, F1-macro = {best_f1_sm:.4f}")
        y_pred_val  = p_val_best.argmax(1)
        val_f1      = f1_score(y_val, y_pred_val, average='macro')
        y_pred_eval = label_encoder.inverse_transform(p_eval_best.argmax(1))

        MODEL_PROBS['temporal_smoothing'] = (p_val_best, p_eval_best, val_f1)
        evaluar_y_guardar('temporal_smoothing', y_pred_val, y_pred_eval,
                          val_score=val_f1, y_true_val=y_val,
                          extra_info=f"Base: {best_model_name}, sigma: {best_sigma}")

---
## SECCIÓN F — Señales lingüísticas adicionales (LightGBM completo)

In [43]:
# Las features de la sección F (ortografía histórica, drift léxico, puntuación,
# POS n-gramas, palabras función, ruido OCR) ya están incorporadas en Xhc_train_sc
# (función extract_handcrafted). Esta celda entrena LightGBM sobre el conjunto
# completo de features incluyendo todas las señales lingüísticas.

if debe_correr('lgbm_linguistic') or debe_correr('ALL'):
    print("\n" + "="*60)
    print("MODELO F: LightGBM con señales lingüísticas históricas completas")
    print("="*60)

    if not LGBM_OK:
        print("  LightGBM no disponible. Instalando...")
        pip_install('lightgbm')
        import lightgbm as lgb
        LGBM_OK = True

    # Combinar: LSA-TF-IDF + handcrafted + OCR quality
    lsa_f = TruncatedSVD(n_components=200, random_state=SEED)
    Xf_lsa_tr  = lsa_f.fit_transform(Xtcomb_train)
    Xf_lsa_val = lsa_f.transform(Xtcomb_val)
    Xf_lsa_ev  = lsa_f.transform(Xtcomb_eval)

    Xff_tr  = np.hstack([Xf_lsa_tr,  Xhc_train_sc]).astype(np.float32)
    Xff_val = np.hstack([Xf_lsa_val, Xhc_val_sc  ]).astype(np.float32)
    Xff_ev  = np.hstack([Xf_lsa_ev,  Xhc_eval_sc ]).astype(np.float32)

    clf_lgbm = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.03, num_leaves=127,
        class_weight='balanced', random_state=SEED, n_jobs=-1,
        min_child_samples=5, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1,
    )
    clf_lgbm.fit(
        Xff_tr, y_train,
        eval_set=[(Xff_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(period=50)],
    )

    y_pred_val  = clf_lgbm.predict(Xff_val)
    val_f1      = f1_score(y_val, y_pred_val, average='macro')
    y_pred_eval = label_encoder.inverse_transform(clf_lgbm.predict(Xff_ev))

    MODEL_PROBS['lgbm_linguistic'] = (clf_lgbm.predict_proba(Xff_val),
                                       clf_lgbm.predict_proba(Xff_ev), val_f1)
    joblib.dump(clf_lgbm, mpath('lgbm_linguistic.pkl'))
    evaluar_y_guardar('lgbm_linguistic', y_pred_val, y_pred_eval,
                      val_score=val_f1, y_true_val=y_val)

---
## SECCIÓN G — Failure Modes y Diagnóstico

In [44]:
if debe_correr('failure_modes') or debe_correr('ALL'):
    print("\n" + "="*60)
    print("DIAGNÓSTICO G: Failure Modes y Contramedidas")
    print("="*60)

    # ── G1. Topic Leakage: evaluación cruzada por género/tópico ──────────
    print("\n[G1] Evaluación por cluster de tópico (proxy de género)...")
    lsa_g = TruncatedSVD(n_components=10, random_state=SEED)
    Xg_val = lsa_g.fit_transform(Xtcomb_train)  # fit en train
    Xg_val_t = lsa_g.transform(Xtcomb_val)

    kmeans_g = KMeans(n_clusters=5, random_state=SEED, n_init=10)
    topic_clusters_val = kmeans_g.fit_predict(Xg_val_t)

    if len(MODEL_PROBS) > 0:
        best_name = max(MODEL_PROBS, key=lambda k: MODEL_PROBS[k][2])
        best_val_probs = MODEL_PROBS[best_name][0]
        best_val_preds = best_val_probs.argmax(1)
        print(f"  Modelo base: {best_name}")
        for cl in range(5):
            mask = topic_clusters_val == cl
            if mask.sum() < 10: continue
            f1_cl = f1_score(y_val[mask], best_val_preds[mask], average='macro', zero_division=0)
            print(f"  Cluster {cl} (n={mask.sum()}): F1-macro = {f1_cl:.4f}")
        overall = f1_score(y_val, best_val_preds, average='macro')
        print(f"  Overall: {overall:.4f} | Max cluster gap puede indicar topic leakage")

    # ── G2. OCR Leakage: evaluar según calidad OCR ────────────────────────
    print("\n[G2] Evaluación por calidad OCR en validación...")
    Xocr_g = ocr_quality_features(X_val.tolist()) if 'ocr_quality_features' in dir() else None
    if Xocr_g is not None:
        ocr_score = Xocr_g[:, 0]  # long_s_ratio como proxy de OCR ruidoso
        high_noise = ocr_score > np.percentile(ocr_score, 75)
        low_noise  = ocr_score < np.percentile(ocr_score, 25)

        if len(MODEL_PROBS) > 0:
            for label, mask in [('Alta calidad (bajo ruido)', low_noise),
                                 ('Baja calidad (alto ruido)', high_noise)]:
                if mask.sum() < 10: continue
                f1_cl = f1_score(y_val[mask], best_val_preds[mask], average='macro', zero_division=0)
                print(f"  {label} (n={mask.sum()}): F1-macro = {f1_cl:.4f}")

    # ── G3. Over-cleaning: comparar con/sin normalización ─────────────────
    print("\n[G3] Efecto de normalización ortográfica (sin-norm vs norm)...")
    for clf_name, X_tr, X_va in [
        ('sin_norm (text_raw)', X_train_raw, X_val_raw),
        ('con_norm (text_norm)', X_train, X_val)
    ]:
        tv = TfidfVectorizer(max_features=30_000, ngram_range=(1,2), sublinear_tf=True, min_df=3)
        Xt = tv.fit_transform(X_tr)
        Xv = tv.transform(X_va)
        clf_g = LogisticRegression(C=0.5, max_iter=300, solver='saga',
                                    multi_class='multinomial', random_state=SEED, n_jobs=-1)
        clf_g.fit(Xt, y_train)
        f1_g = f1_score(y_val, clf_g.predict(Xv), average='macro')
        print(f"  {clf_name}: F1-macro = {f1_g:.4f}")

    # ── G4. Loss Imbalance: pesos de clase vs ordinal loss ────────────────
    print("\n[G4] Impacto de pesos de clase...")
    tv_g = TfidfVectorizer(max_features=30_000, ngram_range=(1,2), sublinear_tf=True, min_df=3)
    Xt_g = tv_g.fit_transform(X_train)
    Xv_g = tv_g.transform(X_val)
    for cw, label in [(None, 'sin pesos'), ('balanced', 'con balanced')]:
        clf_i = LogisticRegression(C=0.5, max_iter=300, solver='saga',
                                    multi_class='multinomial', class_weight=cw,
                                    random_state=SEED, n_jobs=-1)
        clf_i.fit(Xt_g, y_train)
        f1_i = f1_score(y_val, clf_i.predict(Xv_g), average='macro')
        print(f"  class_weight={label}: F1-macro = {f1_i:.4f}")

    # ── G5. Temporal Overfitting: split aleatorio vs temporal ─────────────
    print("\n[G5] Split aleatorio vs temporal (verificación de temporal leakage)...")
    # Ya hemos implementado el split temporal arriba (TEMPORAL_SPLIT flag)
    # Aquí comparamos los dos splits en el mismo modelo
    tv_t = TfidfVectorizer(max_features=20_000, ngram_range=(1,2), sublinear_tf=True, min_df=3)
    clf_t = LogisticRegression(C=0.3, max_iter=300, solver='saga',
                                multi_class='multinomial', class_weight='balanced',
                                random_state=SEED, n_jobs=-1)

    # Split estratificado (ya tenemos)
    Xts_tr = tv_t.fit_transform(X_train)
    Xts_va = tv_t.transform(X_val)
    clf_t.fit(Xts_tr, y_train)
    f1_strat = f1_score(y_val, clf_t.predict(Xts_va), average='macro')

    # Split temporal
    df_s = df_train.sort_values('decade').reset_index(drop=True)
    n = len(df_s)
    df_tr_t = df_s.iloc[:int(0.8*n)]
    df_va_t = df_s.iloc[int(0.8*n):int(0.9*n)]
    X_temp  = df_tr_t['text_norm'].values
    y_temp  = label_encoder.transform(df_tr_t['decade'])
    Xv_temp = df_va_t['text_norm'].values
    yv_temp = label_encoder.transform(df_va_t['decade'])
    Xtt_tr = tv_t.fit_transform(X_temp)
    Xtt_va = tv_t.transform(Xv_temp)
    clf_t.fit(Xtt_tr, y_temp)
    try:
        f1_temp = f1_score(yv_temp, clf_t.predict(Xtt_va), average='macro', zero_division=0)
    except:
        f1_temp = 0.0
    print(f"  Split estratificado: F1-macro = {f1_strat:.4f}")
    print(f"  Split temporal:      F1-macro = {f1_temp:.4f}")
    gap = f1_strat - f1_temp
    if gap > 0.05:
        print(f"  ⚠️  Gap {gap:.4f} > 0.05 → POSIBLE TEMPORAL LEAKAGE en split estratificado")
    else:
        print(f"  ✅ Gap {gap:.4f} ≤ 0.05 → split estratificado es válido")

    print("\n✅ Diagnóstico de failure modes completado.")

---
## SECCIÓN FINAL — Resumen comparativo de scores

In [45]:
print("\n" + "="*70)
print("RESUMEN DE MODELOS ENTRENADOS EN ESTA SESIÓN")
print("="*70)

if _scores_log:
    df_scores = pd.DataFrame(_scores_log).sort_values('f1_macro_val', ascending=False)
    pd.set_option('display.max_colwidth', 50)
    print(df_scores[['modelo', 'f1_macro_val', 'supera_umbral', 'timestamp']].to_string(index=False))

    print(f"\n{'─'*70}")
    n_above = df_scores['supera_umbral'].sum()
    best_row = df_scores.iloc[0]
    print(f"Modelos que superan F1-macro > {VAL_SCORE_UMBRAL}: {n_above} / {len(df_scores)}")
    print(f"Mejor modelo: {best_row['modelo']} — F1-macro = {best_row['f1_macro_val']:.4f}")
    print(f"Log completo: {spath('scores_log.csv')}")
    print(f"Submissions: {SUBMIT_DIR.resolve()}/")
else:
    print("No se han entrenado modelos en esta sesión.")
    print(f"Cambia MODELO_A_EJECUTAR y re-ejecuta las secciones deseadas.")

print("\n" + "="*70)
print("GUÍA DE USO RÁPIDO")
print("="*70)
print("""
1. Para cambiar de modelo:
   → Modifica MODELO_A_EJECUTAR en la Sección 1 y ejecuta la celda del modelo.
   → Ejemplo: MODELO_A_EJECUTAR = 'finetune_bert'

2. Para correr en otra computadora:
   → Copia este notebook + carpeta ./data/ (train.csv, eval.csv).
   → Ejecuta Sección 0 (instalación), Sección 1 (config) y Sección 3 (SETUP).
   → Luego ejecuta el modelo deseado.

3. Predicciones guardadas en: ./submissions/predicciones_{nombre_modelo}.csv
   Formato: columnas 'id' y 'answer' (decade predicha).

4. Para hacer ensemble después de entrenar varios modelos:
   → MODEL_PROBS se llena automáticamente.
   → Ejecuta la sección E (ensembles).

5. Log de scores: ./submissions/scores_log.csv
""")


RESUMEN DE MODELOS ENTRENADOS EN ESTA SESIÓN
      modelo  f1_macro_val  supera_umbral                  timestamp
tfidf_logreg        0.2584          False 2026-05-19T20:52:32.401390
tfidf_logreg        0.2584          False 2026-05-19T21:07:57.085932

──────────────────────────────────────────────────────────────────────
Modelos que superan F1-macro > 0.3: 0 / 2
Mejor modelo: tfidf_logreg — F1-macro = 0.2584
Log completo: submissions\scores_log.csv
Submissions: C:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\submissions/

GUÍA DE USO RÁPIDO

1. Para cambiar de modelo:
   → Modifica MODELO_A_EJECUTAR en la Sección 1 y ejecuta la celda del modelo.
   → Ejemplo: MODELO_A_EJECUTAR = 'finetune_bert'

2. Para correr en otra computadora:
   → Copia este notebook + carpeta ./data/ (train.csv, eval.csv).
   → Ejecuta Sección 0 (instalación), Sección 1 (config) y Sección 3 (SETUP).
   → Luego ejecuta el modelo deseado.

3. Predicciones guardadas en: ./submiss